# Reproducing Kim, Atencio & Schreiner (2020) on CRCNS ac-5

Paper. Kim KX, Atencio CA, Schreiner CE (2020). Stimulus dependent transformations between
synaptic and spiking receptive fields in auditory cortex. *Nature Communications* 11, 1102,
doi:10.1038/s41467-020-14835-7. The claims file (`claims/kim-2020.yaml`) was written from the
version of record (PMC7046699) with its Methods and Supplementary Information.

Dataset. CRCNS ac-5, "In vivo whole-cell recordings to complex sounds in the mouse auditory
cortex", doi:10.6080/K07H1GSS, files under `ac-5/live/` in the public bucket `crcnsarchive`,
read by streaming `data.tar.gz` and `codes.tar.gz` once each. CRCNS data sets are not versioned,
so the version is recorded as `unversioned` (docs/archives.md); there is one published state of
the files (bucket objects dated 2026-02-11, data description version 0.61 of 2020-03-03), and it
is the one used. MD5 sums from the data set's `checksums.md5` for the files read:
`data.tar.gz` 84c11d715bada8da5518fb8da9032505, `codes.tar.gz` 153e23136a08c8b45ecf422b459a620b,
`docs/data_file_information.xlsx` 9515d32985fbbf27f0a6df1a384d8a5f.

Code. The paper links no repository, but the data set ships the paper's Matlab analysis
code (`codes.tar.gz`: `read_data.m`, `peakfinder.m`, `rtwstrfdb.m`, `wcr_strf.m`,
`significant_strf.m`, `fio_to_fio_fit.m`, `ringach_malone_func.m`, `calculate_whole_cell_fsi.m`,
`plot_tuning_curve.m`) and the DMR envelope. The helper `kim2020.py` ports what is used and says,
for each parameter, whether it comes from the paper, the code, the claims file, an assumption
or a standard choice. Where the Methods and the code differ (significance of STRF pixels), the
Methods are followed and the code's choice is shown as a variant.

The table of claims and outcomes is in the summary at the end; every claim is tested except the
two Supplementary Fig. 2 claims and the Supplementary Fig. 3 claim, blocked because the data set
holds only the representative neuron of each (data_missing).

In [ ]:
import time, io, collections
import numpy as np
from scipy import stats
import fsspec, openpyxl
import matplotlib
import matplotlib.pyplot as plt
import kim2020 as K
T0 = time.time()
print(K.DATASET, K.DOI)

In [ ]:
# The DMR envelope (downsampled version used by the paper's wcr_strf.m) and the tone order.
codes = K.load_codes()
env, faxis = codes["env"], codes["faxis"]
envf = K.Envelope(env)
print("envelope", env.shape, "range %.1f to %.1f dB about the mean, s.d. %.2f (40/sqrt(8) = %.2f)"
      % (env.min(), env.max(), env.std(), 40 / np.sqrt(8)))
print("channels %.0f to %.0f Hz, %.4f octave apart" % (faxis[0], faxis[-1], np.log2(faxis[1] / faxis[0])))
print("tones:", len(codes["tone_freq"]), "presentations,", len(np.unique(codes["tone_freq"])),
      "frequencies", np.unique(codes["tone_atten"]), "dB attenuation")

In [ ]:
# One pass over data.tar.gz (about 6 GB, about 5 minutes): tone traces kept, DMR recordings
# reduced to their events as they load.
recs, listing = K.stream_data(log=lambda s: None)
by = collections.defaultdict(list)
for r in recs:
    by[r["kind"]].append(r)
print({k: len(v) for k, v in by.items()}, "%.0f s" % (time.time() - T0))

In [ ]:
# The data set's own table of recordings (docs/data_file_information.xlsx).
with fsspec.open(K.BASE + "docs/data_file_information.xlsx", "rb") as f:
    wb = openpyxl.load_workbook(io.BytesIO(f.read()))
cc_rows = [r for r in wb["current-clamp"].iter_rows(min_row=2, values_only=True) if r[0]]
vc_rows = [r for r in wb["voltage-clamp"].iter_rows(min_row=2, values_only=True) if r[0]]
cc_tone_ids = {str(r[1]) for r in cc_rows}
cc_pairs = {str(r[2]): str(r[1]) for r in cc_rows if r[2]}
vc_exc = {str(r[1]): r[2] for r in vc_rows if r[1]}
vc_inh = {str(r[3]): r[4] for r in vc_rows if r[3]}
vc_pairs = [(str(r[1]), str(r[3])) for r in vc_rows if r[1] and r[3]]
print("current clamp rows", len(cc_rows), "with DMR", len(cc_pairs),
      "| voltage clamp rows", len(vc_rows), "excitation", len(vc_exc), "inhibition", len(vc_inh),
      "both", len(vc_pairs))
print("holding potentials (mV), excitation:", collections.Counter(vc_exc.values()),
      "inhibition:", collections.Counter(vc_inh.values()))

## Data check

In [ ]:
files = {k: sorted(r["file"] for r in v) for k, v in by.items()}
print("kinds and counts:", {k: len(v) for k, v in files.items()})
print("tone files match the table:", set(files["tones"]) == cc_tone_ids)
print("DMR files match the table's paired DMR files:", set(files["cc_dmr"]) == set(cc_pairs))
print("excitation files match:", set(files["vc_exc"]) == set(vc_exc),
      "| inhibition files match:", set(files["vc_inh"]) == set(vc_inh))
print("sampling rates:", collections.Counter(r["fs"] for r in recs))
# processed = raw - smooth(raw, 1e5), as the data descriptor says
d = [r["proc_maxdiff"] for r in recs if "proc_maxdiff" in r]
print("processed vs raw - smooth(raw): %d recordings, largest absolute difference %.2e" % (len(d), max(d)))
print("triggers identical in raw and processed files:", all(r.get("trig_proc_equal", True) for r in recs))
print("recordings where they differ by more than 0.01:", [(r["kind"], r["file"], round(r["proc_maxdiff"], 2), r["proc_ndiff"]) for r in recs if r.get("proc_maxdiff", 0) > 0.01])
# recordings deposited more than once
h = collections.defaultdict(list)
for r in recs:
    h[r["md5_raw"]].append((r["kind"], r["file"]))
dups = [v for v in h.values() if len(v) > 1]
print("identical raw traces:", dups)
# triggers
for k in ("tones", "cc_dmr", "vc_exc", "vc_inh", "supp2", "supp3"):
    nt = collections.Counter(len(np.ravel(r["trig"])) for r in by[k])
    sp = [np.median(np.diff(np.ravel(r["trig"]))) for r in by[k]]
    print(f"{k}: triggers per file {dict(nt)}, median spacing {np.median(sp):.0f} samples")
# ranges of the variables used
tr = np.array([np.median(r["raw"]) for r in by["tones"]])
print("tone recordings, median membrane potential %.1f to %.1f mV" % (tr.min(), tr.max()))
cr = np.array([r["rest"] for r in by["cc_dmr"]])
print("DMR current clamp, median membrane potential %.1f to %.1f mV" % (cr.min(), cr.max()))
ea = np.concatenate([r["amp"] for r in by["cc_dmr"]])
print("DMR current clamp peak amplitudes (processed, mV): %.1f to %.1f, %d peaks" % (ea.min(), ea.max(), len(ea)))
for k in ("vc_exc", "vc_inh"):
    hm = np.array([r["rest"] for r in by[k]])
    a = np.concatenate([r["amp20"] for r in by[k]])
    print(f"{k}: median current {hm.min():.0f} to {hm.max():.0f} pA; peak amplitudes {a.min():.0f} to {a.max():.0f} pA")
# the pre-stimulus segment (D3 reading (a)) and the noise reading (b), per DMR recording
pre_s = np.array([r["trig"][0] / r["fs"] for r in by["cc_dmr"]])
pre_pk = np.array([int((r["pk"] < r["trig"][0]).sum()) for r in by["cc_dmr"]])
print("current clamp DMR: pre-stimulus segment %.1f to %.1f s (median %.1f); peaks in it: median %d (range %d to %d)"
      % (pre_s.min(), pre_s.max(), np.median(pre_s), np.median(pre_pk), pre_pk.min(), pre_pk.max()))
print("  s.d. of the pre-stimulus segment %.2f to %.2f mV (median %.2f); noise s.d. during the DMR %.2f to %.2f mV (median %.2f)"
      % (min(r["pre"] for r in by["cc_dmr"]), max(r["pre"] for r in by["cc_dmr"]), np.median([r["pre"] for r in by["cc_dmr"]]),
         min(r["noise"] for r in by["cc_dmr"]), max(r["noise"] for r in by["cc_dmr"]), np.median([r["noise"] for r in by["cc_dmr"]])))
for k in ("vc_exc", "vc_inh"):
    print(f"  {k}: pre-stimulus s.d. median {np.median([r['pre20'] for r in by[k]]):.1f} pA, noise s.d. median {np.median([r['noise20'] for r in by[k]]):.1f} pA")
# spike detection (D1 (a)): the per-neuron cut, spikes, and peaks near the cut
print("spike cut per DMR neuron (D1 (a)): file, cut mV, spikes, peaks within 10 mV below / above the cut, largest non-spike and smallest spike amplitude")
for r in sorted(by["cc_dmr"], key=lambda r: r["file"]):
    cut = K.spike_cut_gap(r["amp"]); a = r["amp"]
    if np.isfinite(cut):
        print(f"  {r['file']}: {cut:.1f}, {(a > cut).sum()}, {((a > cut - 10) & (a <= cut)).sum()} / {((a > cut) & (a <= cut + 10)).sum()}, "
              f"{a[a <= cut].max():.1f} / {a[a > cut].min():.1f}; rate-of-rise spikes {r['dvdt'].sum()}")
    else:
        print(f"  {r['file']}: no gap, 0 spikes; largest peak {a.max():.1f}; rate-of-rise spikes {r['dvdt'].sum()}")
dates_cc = sorted({r[0].date() for r in cc_rows})
dates_vc = sorted({r[0].date() for r in vc_rows})
# the archive's members: data.tar.gz listing by folder, and every member under Supplementary/
top = collections.Counter("/".join(n.split("/")[:3]) for n, _ in listing if n.endswith(".mat"))
print("data.tar.gz: %d members, .mat files by folder:" % len(listing), dict(top))
for n, sz in listing:
    if "Supplementary" in n:
        print("  ", n, sz, "bytes")
print("codes.tar.gz: %d members:" % len(codes["listing"]), [n.split("/")[-1] for n, _ in codes["listing"]])
print("recording dates: current clamp", len(dates_cc), "| voltage clamp", len(dates_vc),
      "| no animal identifier in the files or the table")

Clock check. The tone triggers and the DMR block triggers share the response's sampling
clock (both are channels of the same file). That a response is locked to them is checked by a
contrast: for tones, the mean membrane potential 5 to 50 ms after onset minus the 50 ms before,
for loud (60 to 70 dB SPL) against quiet (0 to 10 dB SPL) tones; for the DMR, the share of STRF
pixels beyond the circular-shift threshold (5% expected by chance) and the latency of the
strongest pixel.

In [ ]:
loud, quiet = [], []
for r in by["tones"]:
    raw = r["raw"].astype(float); tg = np.ravel(r["trig"]).astype(int)
    lvl = 70 - codes["tone_atten"]
    d = np.array([raw[t + 50:t + 500].mean() - raw[t - 500:t].mean() for t in tg])
    loud.append(d[lvl >= 60].mean()); quiet.append(d[lvl <= 10].mean())
loud, quiet = np.array(loud), np.array(quiet)
print("tones: post - pre, loud %.2f mV, quiet %.2f mV; loud > quiet in %d of %d neurons, paired t p = %.1e"
      % (loud.mean(), quiet.mean(), (loud > quiet).sum(), len(loud), stats.ttest_rel(loud, quiet).pvalue))

The lines of the paper's Matlab code that the helper ports, printed from `codes.tar.gz`, so
that the port can be checked: the STRF script and the envelope file it names, the envelope
scaling, the nonlinearity function, the analytic significance threshold and the FSI.

In [ ]:
import re
def show_lines(fname, pattern=None, maxl=40):
    lines = codes["code"][fname].splitlines()
    for k, l in enumerate(lines):
        if (pattern is None and l.strip() and not l.strip().startswith("%")) or (pattern and re.search(pattern, l) and not l.strip().startswith("%")):
            print(f"{fname}:{k + 1}: {l.rstrip()}")
show_lines("wcr_strf.m")
show_lines("read_entire_spr_file.m", r"MdB|RMSP|reshape")
show_lines("ringach_malone_func.m", r"fx =|exp\(")
show_lines("significant_strf.m", r"siglevel|sigma =")
show_lines("calculate_whole_cell_fsi.m", r"plevel =|wstrfstat\(rf|hist\(p|CDFi|FSI1=")
show_lines("read_data.m", r"peakfinder|smooth")

## Tones (Fig. 1)

TRFs per neuron from the 66 tone recordings. Details the claims file leaves open are taken as it
lists them (D3, D14 to D16); the readings are run side by side.

In [ ]:
def tone_table(sd_reading="pooled"):
    rows = []
    for r in sorted(by["tones"], key=lambda r: r["file"]):
        raw = r["raw"].astype(float)
        t = K.tone_analysis(raw, r["trig"], codes["tone_freq"], codes["tone_atten"], sd_reading=sd_reading)
        amps, rises = K.psp_amp_kinetics(t["post"], t["resp"], t["spike"])
        # decay 90-10% after the peak, followed to 250 ms, on tones with no spike in the 250 ms
        dec = []; n_nodecay = 0; n_latespk = 0
        tg = np.ravel(r["trig"]).astype(int)
        for k in np.nonzero(t["resp"] & ~t["spike"])[0]:
            if np.any((t["spk"] >= tg[k]) & (t["spk"] < tg[k] + 2500)):
                n_latespk += 1; continue
            y = raw[tg[k]:tg[k] + 2500] - t["pre_mean"][k]
            i = int(np.argmax(y[:1000])); a = y[i]
            j90 = np.nonzero(y[i:] < 0.9 * a)[0]; j10 = np.nonzero(y[i:] < 0.1 * a)[0]
            if len(j90) and len(j10):
                dec.append((j10[0] - j90[0]) / 10.0)
            else:
                n_nodecay += 1
        # D14 (a) as the amplitude reference: peak minus the median potential of the recording
        rest_a = float(np.median(raw))
        ka = np.nonzero(t["resp"] & ~t["spike"])[0]
        amp_ref_a = np.array([raw[tg[k]:tg[k] + K.RESP_MS * 10].max() - rest_a for k in ka])
        pre_means = t["pre_mean"]
        row = dict(file=r["file"], rest_a=float(np.median(raw)), rest_b=float(pre_means.mean()),
                   amp_max=amps.max() if len(amps) else np.nan, amp_mean=amps.mean() if len(amps) else np.nan,
                   rise=np.mean(rises) if len(rises) else np.nan, decay=np.mean(dec) if dec else np.nan,
                   amp_max_ra=amp_ref_a.max() if len(ka) else np.nan, amp_mean_ra=amp_ref_a.mean() if len(ka) else np.nan,
                   n_nodecay=n_nodecay, n_latespk=n_latespk,
                   nresp=int(t["resp"].sum()), sd=float(t["sd"].mean()), t=t)
        for reg in "ab":
            for bw in "ab":
                s = K.trf_summary(t, region=reg, bw=bw)
                for k, v in s.items():
                    row[f"{k}_{reg}{bw}"] = v
        rows.append(row)
    return rows

tones = tone_table("pooled")
tones_pt = tone_table("pertone")
print("neurons", len(tones), "| baseline s.d. (pooled) %.2f to %.2f mV" % (min(r['sd'] for r in tones), max(r['sd'] for r in tones)))
print("responsive tones per neuron: %d to %d (pooled s.d.), %d to %d (per-tone s.d.)"
      % (min(r['nresp'] for r in tones), max(r['nresp'] for r in tones),
         min(r['nresp'] for r in tones_pt), max(r['nresp'] for r in tones_pt)))
print("decay: tones left out for a spike within 250 ms %d, tones whose decay never reaches 10%% in 250 ms %d; neurons with no decay value %d"
      % (sum(r["n_latespk"] for r in tones), sum(r["n_nodecay"] for r in tones), sum(not np.isfinite(r["decay"]) for r in tones)))
thr = collections.Counter(r["thr_aa"] for r in tones)
print("threshold level (dB SPL) of the TRFs:", dict(thr), "(paper, Methods: 0 dB SPL in 63, 10 dB SPL in 3)")

In [ ]:
def within(v, e, tol):
    return bool(np.isfinite(v) and abs(v - e) <= tol + 1e-12)

def show(name, v, e, tol, unit=""):
    ok = within(v, e, tol)
    print(f"  {name}: {v:.3g} {unit} (expected {e} +- {tol}) -> {'within' if ok else 'outside'}")
    return ok

### cc-cohort-counts
> Current-clamp recordings with pure tones were made in 66 neurons from 41 mice, and 40 of these
> neurons were also recorded with the DMR stimulus.

In [ ]:
n_t, n_d = len(by["tones"]), len(by["cc_dmr"])
paired = sum(1 for d in files["cc_dmr"] if d in cc_pairs and cc_pairs[d] in files["tones"])
print("tone recordings", n_t, "| DMR recordings", n_d, "| DMR recordings whose neuron has a tone recording", paired)
print("mice: the files and the table carry no animal identifier; recording dates %d (current clamp)" % len(dates_cc))

**Result: reproduced.** The data set holds 66 tone recordings and 40 DMR recordings, each paired
with a tone recording of the same neuron by the data set's table. The number of mice (41) cannot
be counted: no animal identifier is given; the recordings fall on the number of dates printed above.

### vc-cohort-counts
> Voltage-clamp DMR recordings were made in 31 neurons from 24 mice, with 27 excitatory and 12
> inhibitory recordings, both components in 8 neurons.

In [ ]:
print("excitatory recordings", len(by["vc_exc"]), "| inhibitory", len(by["vc_inh"]),
      "| neurons in the table", len(vc_rows), "| with both", len(vc_pairs))
print("recording dates (voltage clamp):", len(dates_vc), "; no animal identifier")

**Result: reproduced.** 27 excitatory and 12 inhibitory recordings from 31 neurons, 8 with
both. Mice cannot be counted (no identifier); the number of recording dates is printed above.

### fig1-psp-amplitude-kinetics
> For pure tones the maximum PSP amplitude was 16.3 +- 4.8 mV, the 10 to 90% rise time 30 +- 17
> ms and the 90 to 10% decay time 86 +- 20 ms (mean +- s.d., n = 66).

Readings: D16 (a) the largest tone-evoked PSP per neuron, (b) the mean over its responsive
tones; D3 baseline s.d. pooled over the pre-tone segments or per tone. Amplitudes are taken from
the 50-ms pre-tone mean of each tone (D14 reading (b) as the local reference).

In [ ]:
for lab, T in (("pooled s.d.", tones), ("per-tone s.d.", tones_pt)):
    a_max = np.array([r["amp_max"] for r in T]); a_mean = np.array([r["amp_mean"] for r in T])
    rise = np.array([r["rise"] for r in T]); dec = np.array([r["decay"] for r in T])
    print(lab, "| neurons with a responsive PSP without spike:", np.isfinite(a_max).sum())
    for name, v in (("D16 (a) largest PSP", a_max), ("D16 (b) mean PSP", a_mean)):
        print(f"  {name}: {np.nanmean(v):.1f} +- {np.nanstd(v):.1f} mV (range {np.nanmin(v):.1f} to {np.nanmax(v):.1f})")
    print(f"  rise 10-90%: {np.nanmean(rise):.1f} +- {np.nanstd(rise):.1f} ms; decay 90-10%: {np.nanmean(dec):.1f} +- {np.nanstd(dec):.1f} ms")
    ok = [show("max amplitude (a)", np.nanmean(a_max), 16.3, 4.8, "mV"), show("rise", np.nanmean(rise), 30, 17, "ms"),
          show("decay", np.nanmean(dec), 86, 20, "ms")]
    okb = show("mean amplitude (b)", np.nanmean(a_mean), 16.3, 4.8, "mV")
    print("  all three within tolerance: reading (a)", all(ok), "| reading (b)", okb and ok[1] and ok[2])
    ra, rb = np.array([r["amp_max_ra"] for r in T]), np.array([r["amp_mean_ra"] for r in T])
    print(f"  amplitude from the median potential (D14 (a) as reference): largest {np.nanmean(ra):.1f} +- {np.nanstd(ra):.1f} mV "
          f"(within {within(np.nanmean(ra), 16.3, 4.8)}), mean {np.nanmean(rb):.1f} +- {np.nanstd(rb):.1f} mV (within {within(np.nanmean(rb), 16.3, 4.8)})")

**Result: reproduced, under reading (b) of D16 and the pooled baseline s.d.** With the mean PSP per neuron, the amplitude is 14.0 +- 6.3 mV, the rise time 30.3 +- 9.5 ms and the decay 91.5 +- 16.8 ms, all within tolerance. Taking the amplitude from the median potential of the recording (D14 (a)) in place of the pre-tone mean gives 13.0 mV, also within. The largest PSP per neuron (reading (a)) is 25.2 +- 11.6 mV (23.8 from the median potential), outside 16.3 +- 4.8. With a per-tone baseline s.d., the mean amplitude is 10.3 mV and no reading matches all three values. The decay leaves out the 307 tones with a spike within 250 ms; 919 tones whose decay does not reach 10% within 250 ms give no decay value, and no neuron is left without one.

### fig1c-resting-potential-range
> Resting membrane potentials of the 66 neurons ranged from -85 to -59 mV.

D14 readings: (a) the median membrane potential over the tone recording; (b) the mean over the
pre-tone baselines.

In [ ]:
for lab, key in (("(a) median", "rest_a"), ("(b) pre-tone mean", "rest_b")):
    v = np.array([r[key] for r in tones])
    print(f"{lab}: {v.min():.1f} to {v.max():.1f} mV, mean {v.mean():.1f}; above -50 mV: {(v > -50).sum()}")
    lo_ok = show("most negative", v.min(), -85, 8.5, "mV"); hi_ok = show("least negative", v.max(), -59, 5.9, "mV")
    print("  both within:", lo_ok and hi_ok)

**Result: reproduced.** The median membrane potentials of the 66 tone recordings span -84.7 to -55.1 mV (reading (a)), and the pre-tone means span -83.8 to -53.7 mV (reading (b)). Both ranges fall within -85 +- 8.5 and -59 +- 5.9. No recording rests above -50 mV.

### fig1d-ndr-distribution
> The distribution of normalized-driven ratios resembles an exponential decay, with many neurons
> near 0 and very few near 1; 31 of the 66 neurons have a ratio of at least 0.3.

NDR = tones with a spike over tones with a PSP within the TRF region, one spike per tone
(Methods). Spikes: peakfinder events above the per-neuron gap of the amplitude histogram (D1
(a)). D15 region readings (a) responsive tones, (b) the outline; D3 pooled or per-tone s.d.

In [ ]:
def ndr_report(T, key):
    v = np.array([r[key] for r in T])
    h, _ = np.histogram(v, bins=np.linspace(0, 1, 11))
    n03 = int((v >= 0.3).sum())
    crit = (h.argmax() == 0) and ((v > 0.7).mean() < 0.1) and within(n03, 31, 3.1)
    print(f"  {key}: bins of 0.1 {h.tolist()}; >= 0.3: {n03}; > 0.7: {(v > 0.7).sum()}; criterion met: {crit}")
    return v
for lab, T in (("pooled s.d.", tones), ("per-tone s.d.", tones_pt)):
    print(lab)
    for key in ("ndr_aa", "ndr_ba"):
        ndr_report(T, key)
print("spikes per tone recording (all, not only in responses): median",
      int(np.median([r["t"]["nspk"] for r in tones])), "; recordings with no spike:", sum(r["t"]["nspk"] == 0 for r in tones))

**Result: reproduced, with the pooled baseline s.d. and the responsive tones as the TRF region (D15 (a)), an assumption.** Under region (a), the 0 to 0.1 bin holds the most neurons (19), 6 of 66 lie above 0.7 (9%), and 31 neurons have an NDR of at least 0.3. Region (b), the outline, also counts spikes at tones inside it whose PSP onset was not detected; it gives 33 neurons at 0.3 or more but 11 of 66 (17%) above 0.7, so the criterion fails there. Region (a) is adopted because it counts spikes and PSPs over the same tones. With a per-tone baseline s.d., 18 (region (a)) or 21 (region (b)) neurons reach 0.3.

### fig1e-ndr-vs-rest-weak
> Normalized-driven ratios and resting membrane potentials were weakly and not significantly
> correlated, more depolarized neurons tending to produce more spikes.

In [ ]:
for lab, T in (("pooled s.d.", tones), ("per-tone s.d.", tones_pt)):
    for rk in ("rest_a", "rest_b"):
        for nk in ("ndr_aa", "ndr_ba"):
            x = np.array([r[rk] for r in T]); y = np.array([r[nk] for r in T])
            ok = np.isfinite(x) & np.isfinite(y)
            rr, p = stats.pearsonr(x[ok], y[ok])
            print(f"{lab}, {rk}, {nk}: n = {ok.sum()}, r = {rr:.3f}, p = {p:.3g}; within 0.18 +- 0.018: {within(rr, 0.18, 0.018)}")

**Result: reproduced, under D14 (b) with region (a), or D14 (a) with region (b), with the pooled baseline s.d.** With the pre-tone mean as the resting potential and region (a), r = 0.182 and p = 0.144 (n = 66); with the median potential and region (b), r = 0.187 and p = 0.134. The median potential with region (a) gives r = 0.116 (p = 0.355), positive but outside tolerance; the pre-tone mean with region (b) gives r = 0.244 with p = 0.0483, significant. The per-tone baseline s.d. gives r = 0.035 to 0.098.

### fig1fgh-trf-bandwidth-narrowing
> In the 31 neurons with normalized-driven ratio of at least 0.3, the tonal PSP bandwidth30 was
> 1.38 +- 0.33 octaves and the spike bandwidth30 1.16 +- 0.34 octaves, spikes being significantly
> narrower (paired t-test), with a median spike/PSP ratio of about 0.9, about 16% narrower.

Bandwidth30 at the threshold level + 30 dB (Methods, Stimuli); D15 readings (a) highest over
lowest responsive frequency, (b) the contiguous range around CF; region readings as for the NDR.

In [ ]:
for lab, T in (("pooled s.d.", tones), ("per-tone s.d.", tones_pt)):
    for reg in "ab":
        for bw in "ab":
            sel = [r for r in T if r[f"ndr_{reg}a"] >= 0.3]
            p_ = np.array([r[f"bw_psp_{reg}{bw}"] for r in sel]); s_ = np.array([r[f"bw_spk_{reg}{bw}"] for r in sel])
            ok = np.isfinite(p_) & np.isfinite(s_)
            if ok.sum() < 3:
                print(lab, reg, bw, "too few"); continue
            t = stats.ttest_rel(s_[ok], p_[ok])
            with np.errstate(divide="ignore", invalid="ignore"):
                ratio = np.median(s_[ok] / p_[ok])
            good = within(p_[ok].mean(), 1.38, 0.33) and within(s_[ok].mean(), 1.16, 0.34) and within(ratio, 0.89, 0.089)
            print(f"{lab}, region {reg}, bandwidth {bw}: NDR>=0.3 n = {len(sel)}, both bandwidths n = {ok.sum()}; "
                  f"PSP {p_[ok].mean():.2f} +- {p_[ok].std():.2f}, spike {s_[ok].mean():.2f} +- {s_[ok].std():.2f} oct, "
                  f"median ratio {ratio:.2f}; paired t = {t.statistic:.2f}, p = {t.pvalue:.2g}; values within: {good}")

**Result: partially reproduced.** In the 31 neurons with NDR of at least 0.3 under region (a) (28 with both bandwidths), the spike bandwidth30 (1.49 +- 0.94 octaves) is narrower than the PSP bandwidth30 (1.94 +- 0.76): paired t p = 0.00037, median ratio 0.90. The spike mean and the ratio fall within tolerance; the PSP mean does not (expected 1.38 +- 0.33). Region (b) selects 33 neurons (31 with both): PSP 1.94 and spike 1.48 octaves, median ratio 0.94, p = 0.00027, the PSP mean again outside. The contiguous reading of D15 (b) breaks the single-presentation TRFs into isolated tones (PSP 0.20 octaves).

## DMR, current clamp (Figs. 2 to 6, Table 1)

Events: peakfinder with selectivity 3 mV (the paper's code) on the data set's processed traces
(raw minus a 10-s running mean; the data description calls them the derived data used for the
figures); spikes by the per-neuron gap of the amplitude histogram (D1 (a)); non-spiking events
kept above 4 x the baseline s.d. (D3: the main reading is the noise of event-free 50-ms segments
during the DMR); large and small PSPs split at the KDE trough when bimodal, else at the median.
STRFs are event-triggered averages of the downsampled envelope (5 ms x 64 channels, 0 to 195 ms
before the event, the paper's `wcr_strf.m` and `rtwstrfdb.m`); pixels significant at p < 0.05
against 100 circularly shifted event trains (Methods; D6).

Which neurons have significant STRFs follows the claims file's D7 readings, which here select
the same 37 neurons. Within an STRF, the bands used for BF, bandwidth, latency, duration and
double peaks are subfields kept by a cluster test on the same null (assumption, in the helper):
without it, chance regions make 36 to 40 of the 40 STRFs of each type double-peaked (printed
under fig3d-double-peaked-counts).

In [ ]:
cfs = {d: next(r["cf_aa"] for r in tones if r["file"] == t) for d, t in cc_pairs.items()}
cc = K.cc_neurons(by["cc_dmr"], envf, faxis, cfs=cfs)
print("neurons", len(cc))
ne = {c: np.array([n["n_events"][c] for n in cc]) for c in ("spike", "large", "small", "all")}
for c, v in ne.items():
    print(f"{c}: events per neuron {v.min()} to {v.max()}, median {np.median(v):.0f}")
print("baseline s.d. (noise reading): %.2f to %.2f mV" % (min(n['ev']['sd'] for n in cc), max(n['ev']['sd'] for n in cc)))
print("%.0f s so far" % (time.time() - T0))

### fig2-psp-histogram-modality
> Non-spiking PSP peak amplitude distributions were bimodal in 60% (24/40) and unimodal in 40%
> (16/40) of the neurons recorded with the DMR.

In [ ]:
rows = []
for sd in ("noise", "quiet", "pre"):
    for mod in ("kde", "dip"):
        for amp in ("proc", "rest"):
            cfg = dict(sd=sd, spike="gap", modality=mod, amp=amp)
            b = sum(K.split_events(r, cfg)["bimodal"] for r in by["cc_dmr"])
            kept = np.median([(K.split_events(r, cfg)["large"] | K.split_events(r, cfg)["small"]).sum() for r in by["cc_dmr"]])
            rows.append((sd, mod, amp, b, 40 - b, kept))
            print(f"D3 {sd:5s} D4 {mod} D2 {amp}: bimodal {b}, unimodal {40 - b} (median non-spiking events kept {kept:.0f}); "
                  f"within 24 +- 2.4 and 16 +- 1.6: {within(b, 24, 2.4) and within(40 - b, 16, 1.6)}")

**Result: not reproduced.** Under the main reading (noise s.d., KDE, the code's amplitude), 17 of 40 neurons are bimodal and 23 unimodal. The closest variant (rest-referenced amplitudes) gives 21 and 19, still outside 24 +- 2.4. The dip test finds 0 to 9 bimodal neurons, depending on the reading. A baseline s.d. from the pre-stimulus trace (D3 (a)) leaves a median of 0 or 1 non-spiking events per neuron: the pre-stimulus segment (12.5 to 18.7 s) holds 11 to 115 peaks, and its s.d. (median 6.32 mV) is about five times the event-free noise during the DMR (median 1.28 mV), as the data check prints.

### fig2-strf-significance-groups
> Of the 40 DMR neurons, 75% (30/40) had significant STRF subfields for spikes, large PSPs and
> small PSPs, and 25% (10/40) lacked significant spike STRF subfields but had significant PSP
> STRF subfields.

D7 readings: (a) at least one significant positive pixel; (b) a contiguous significant positive
region of at least 2 pixels (no cluster test). Variants: subfields kept by a cluster test (the
first round's main reading); spikes by the rate-of-rise criterion (D1 (b)); the code's analytic
significance (p = 0.005, `plot_strf_single.m`).

In [ ]:
def groups(neurons, key):
    all3 = [n for n in neurons if all(n["strf"][c][key] for c in ("spike", "large", "small"))]
    psp_only = [n for n in neurons if not n["strf"]["spike"][key] and (n["strf"]["large"][key] or n["strf"]["small"][key])]
    neither = [n for n in neurons if not any(n["strf"][c][key] for c in ("spike", "large", "small"))]
    return all3, psp_only, neither
# D7 as the claims file defines it: (a) any significant positive pixel, (b) a contiguous significant
# positive region of at least 2 pixels, both on the circular-shift null at p < 0.05, no cluster test
cc_nc = K.cc_neurons(by["cc_dmr"], envf, faxis, cfs=cfs, cluster=False)
for key, lab in (("sig_any", "D7 (a) any pixel"), ("sig_sub", "D7 (b) region of >= 2 pixels")):
    a3, po, ne_ = groups(cc_nc, key)
    print(f"{lab}: all three {len(a3)}, PSP but not spike {len(po)}, none {len(ne_)}, other {40 - len(a3) - len(po) - len(ne_)}; "
          f"within 30 +- 3 and 10 +- 1: {within(len(a3), 30, 3) and within(len(po), 10, 1)}")
print("D7 (a) and (b) select the same neurons:", [n["file"] for n in groups(cc_nc, "sig_any")[0]] == [n["file"] for n in groups(cc_nc, "sig_sub")[0]])
a3, po, ne_ = groups(cc, "sig_sub")
print(f"variant, subfields kept by a cluster test on the same null (assumption): all three {len(a3)}, PSP but not spike {len(po)}, none {len(ne_)}, other {40 - len(a3) - len(po) - len(ne_)}")
cc_dvdt = K.cc_neurons(by["cc_dmr"], envf, faxis, cfg=dict(K.MAIN, spike="dvdt"), cfs=cfs)
cc_an = K.cc_neurons(by["cc_dmr"], envf, faxis, cfs=cfs, sig_reading="analytic", cluster=False)
for nm, nn in (("spikes by rate of rise (D1 (b)), cluster test", cc_dvdt), ("the code's analytic p = 0.005", cc_an)):
    for key in ("sig_any", "sig_sub"):
        a3, po, ne_ = groups(nn, key)
        print(f"{nm}, {key}: all three {len(a3)}, PSP but not spike {len(po)}")
# the group the Table 1 claims use: the 37 neurons D7 (a) and (b) both select. Their STRF measures
# (significant STRF for RTF, nonlinearity and FSI; bands from subfields kept by the cluster test
# for BF, bandwidth, latency and double peaks) come from `cc`.
fc = {n["file"] for n in groups(cc_nc, "sig_any")[0]}
G = [n for n in cc if n["file"] in fc]
GC = groups(cc, "sig_sub")[0]           # variant: cluster test, 15 neurons
GN = groups(cc_an, "sig_sub")[0]        # variant: the code's analytic significance, 29 neurons
GA = G
GROUPS = [("main, D7 (a) = (b)", G), ("code's analytic p = 0.005", GN), ("cluster test", GC)]
ns3 = np.array([n["n_events"]["spike"] for n in G])
print("spikes per neuron in the main group: median %d (min %d)" % (np.median(ns3), ns3.min()))
print("bimodal in the main group:", sum(n['ev']['bimodal'] for n in G), "of", len(G))
for gl, gg in GROUPS:
    print(f"group {gl}: n = {len(gg)}")

**Result: not reproduced.** D7 (a) (any significant positive pixel) and D7 (b) as the claims file defines it (a contiguous significant positive region of at least 2 pixels) select the same neurons: 37 with STRFs for all three types and 3 with PSP but not spike STRFs, against 30 and 10. Two variants come closer on one count but not on both: the code's analytic significance (p = 0.005, `significant_strf.m`) gives 29 and 6, and subfields kept by a cluster test on the circular-shift null (the main reading of the first round, an assumption) give 15 and 11, with 13 in other patterns. The rate-of-rise spike criterion changes the counts by at most two. The tests that follow use the 37 neurons both D7 readings select, with the 29 and the 15 shown as variants.

### fig2-psp-amplitude-ranges
> In the 30 neurons with significant STRFs for all event types, the mean amplitudes of large PSPs
> ranged from 13.8 to 28.8 mV and of small PSPs from 7.2 to 11.5 mV, two ranges that do not overlap.

In [ ]:
evs = {r["file"]: r for r in by["cc_dmr"]}
for gl, gg in GROUPS:
    for amp in ("proc", "rest"):
        L, S = [], []
        for n in gg:
            ev = K.split_events(evs[n["file"]], dict(K.MAIN, amp=amp))
            L.append(ev["amp"][ev["large"]].mean()); S.append(ev["amp"][ev["small"]].mean())
        L, S = np.array(L), np.array(S)
        ok = [within(L.min(), 13.8, 1.38), within(L.max(), 28.8, 2.88), within(S.min(), 7.2, 0.72), within(S.max(), 11.5, 1.15)]
        print(f"{gl}, D2 {amp}: n = {len(L)}; large {L.min():.1f} to {L.max():.1f} mV, small {S.min():.1f} to {S.max():.1f} mV; "
              f"bounds within {ok}; ranges overlap: {S.max() >= L.min()}")

**Result: not reproduced.** In the 37 neurons, the mean large-PSP amplitudes range from 6.9 to 21.7 mV and the small-PSP means from 3.3 to 13.0 mV, so the two ranges overlap; rest-referenced amplitudes give 7.7 to 22.8 and 3.7 to 11.9 mV. Both variant groups overlap too (29 neurons: 10.4 to 21.7 and 4.1 to 13.0 mV; 15 neurons: 10.4 to 21.7 and 5.5 to 13.0 mV). Only the highest rest-referenced small-PSP mean falls within its tolerance.

### Double peaks, harmonic ratios and bandwidths (Fig. 3)

### fig3d-double-peaked-counts
> Half of the 40 neurons (20/40) had double-peaked STRFs for at least one event type; double
> peaks were found in 18/40 PSP STRFs and in only 7/40 (17.5%) spike STRFs, about 61% fewer, and
> in 14/40 small-PSP STRFs.

D9 readings: (a) two frequency-separated significant subfields (bands); (b) two maxima of the
frequency profile separated by a dip below half of the smaller. For the 20/40, (a) any event
type, (b) the all-event STRF.

In [ ]:
def dbl(n, c, rd):
    m = n["strf"][c]
    return (m.get("nbands", 0) == 2) if rd == "a" else bool(m.get("double_b", False))
for vl, nn, rds in (("subfields by the cluster test", cc, "ab"), ("no cluster test (regions of >= 2 pixels)", cc_nc, "a"),
                    ("the code's analytic p = 0.005", cc_an, "a")):
    for rd in rds:
        sp = sum(dbl(n, "spike", rd) for n in nn); lg = sum(dbl(n, "large", rd) for n in nn)
        sm = sum(dbl(n, "small", rd) for n in nn); al = sum(dbl(n, "all", rd) for n in nn)
        psp = sum(dbl(n, "large", rd) or dbl(n, "small", rd) for n in nn)
        anyt = sum(any(dbl(n, c, rd) for c in ("spike", "large", "small", "all")) for n in nn)
        ok_a = within(anyt, 20, 2) and within(psp, 18, 1.8) and within(sp, 7, 0.7) and within(sm, 14, 1.4) and sp < psp
        ok_b = within(al, 20, 2) and within(psp, 18, 1.8) and within(sp, 7, 0.7) and within(sm, 14, 1.4) and sp < psp
        print(f"{vl}, D9 ({rd}): spike {sp}, large {lg}, small {sm}, PSP (large or small) {psp}, all-event {al}, any type {anyt}; "
              f"criterion with 20 = any type: {ok_a}; with 20 = all-event: {ok_b}")

**Result: not reproduced.** With bands as frequency-separated subfields kept by the cluster test (D9 (a)), 16 spike, 9 large-PSP and 8 small-PSP STRFs are double-peaked; 15 neurons have a double-peaked PSP STRF, 17 a double-peaked all-event STRF, and 28 one of any type. Spikes are double-peaked about as often as PSPs, not less often. Without the cluster test (regions of at least 2 pixels), 36 to 40 STRFs of every type are double-peaked; with the code's analytic significance, 30 to 33. The profile reading (b) makes 35 or more of 40 spike, large-PSP and small-PSP STRFs double-peaked. No reading matches.

### fig3e-harmonic-ratios
> Peak frequency ratios of the 20 double-peaked all-event STRFs fell within 4.5% of five
> whole-number ratios (1.25, 1.33, 1.5, 1.67, 2), from 5/4 to 2/1, with 1.67 the most common (6/20).

In [ ]:
cats = np.array([5 / 4, 4 / 3, 3 / 2, 5 / 3, 2.0])
for vl, nn in (("cluster test", cc), ("no cluster test", cc_nc), ("code's analytic p = 0.005", cc_an)):
    ratios = np.array([n["strf"]["all"]["peak_freqs"][1] / n["strf"]["all"]["peak_freqs"][0]
                       for n in nn if n["strf"]["all"].get("nbands", 0) == 2])
    print(f"{vl}: double-peaked all-event STRFs: {len(ratios)}; ratios", np.round(np.sort(ratios), 2))
    inside = ratios[(ratios >= 1.25 * 0.955) & (ratios <= 2 * 1.045)]
    print("  ratios within 4.5% of the 1.25 to 2 range:", len(inside), "of", len(ratios))
    asg = np.argmin(np.abs(ratios[:, None] - cats[None, :]), axis=1)
    cnt = np.bincount(asg, minlength=5)
    err = [abs(ratios[asg == k].mean() - cats[k]) / cats[k] * 100 if cnt[k] else np.nan for k in range(5)]
    print("  counts per category", dict(zip(np.round(cats, 2).tolist(), cnt.tolist())), "; percent error of category means", np.round(err, 1))
    ok = (cnt.argmax() == 3) and within(cnt[3], 6, 0.6) and np.nanmax(err) <= 4.95
    print("  1.67 most common:", cnt.argmax() == 3, "| count within 6 +- 0.6:", within(cnt[3], 6, 0.6),
          "| all category means within 4.95%:", np.nanmax(err) <= 4.95, "-> criterion met:", ok)
    if vl == "cluster test":
        # ratios inside the five categories alone (the verifier's point 4: far-apart bands)
        ins = ratios[(ratios >= 1.25 * 0.955) & (ratios <= 2 * 1.045)]
        a2 = np.argmin(np.abs(ins[:, None] - cats[None, :]), axis=1); c2 = np.bincount(a2, minlength=5)
        e2 = [abs(ins[a2 == k].mean() - cats[k]) / cats[k] * 100 if c2[k] else np.nan for k in range(5)]
        print("  ratios inside 1.25 to 2 only: counts", c2.tolist(), "; percent errors", np.round(e2, 1))

**Result: not reproduced.** With the cluster test, of 17 double-peaked all-event STRFs, 12 have ratios within the 1.25 to 2 range widened by 4.5%. The 1.33, 1.5 and 1.67 categories hold 4 each and 2 holds 5, so 1.67 is not the most common; five ratios (2.47 to 15.07) lie above the range, which puts the 2 category's error at 279%. Counting only the 12 ratios inside the range, the category errors are 6.2, 1.2 and 0.6%, and 1.33 still misses 4.95%. Without the cluster test all 40 all-event STRFs are double-peaked, and with the code's analytic significance 33; in both, 2 is the most common category (22 and 16). On the 0.1-octave envelope grid, ratios can only take discrete values (1.42, 1.52, 1.63, 1.74).

### fig3f-bandwidth-single-vs-double
> All-event STRF bandwidths had a median of 0.28 octaves in single-peaked and 0.96 octaves in
> double-peaked neurons.

In [ ]:
for vl, nn in (("cluster test", cc), ("code's analytic p = 0.005", cc_an)):
    for grp in ("a", "b"):
        for bwk in ("total_a", "total_b", "single_main"):
            d_, s_ = [], []
            for n in nn:
                m = n["strf"]["all"]
                if bwk not in m:
                    continue
                isd = any(dbl(n, c, "a") for c in ("spike", "large", "small", "all")) if grp == "a" else dbl(n, "all", "a")
                (d_ if isd else s_).append(m[bwk])
            print(f"{vl}, grouping ({grp}), bandwidth {bwk}: single n = {len(s_)} median {np.median(s_):.2f}, "
                  f"double n = {len(d_)} median {np.median(d_):.2f} oct; within: {within(np.median(s_), 0.28, 0.028) and within(np.median(d_), 0.96, 0.096)}")

**Result: reproduced, under grouping (a) and the total bandwidth D8 (a), with subfields kept by the cluster test (assumption).** The median all-event bandwidth is 0.30 octaves in the 9 neurons with no double-peaked STRF and 0.90 octaves in the 26 with one; both are within tolerance. It is the only one of the 12 combinations printed that matches. The groups are 9 and 26, not 20 and 20; grouping (b) gives 0.35 and 1.00; the half-maximum width (D8 (b)) and the main band alone give 0.20 to 0.40 in every group; with the code's analytic significance only 1 or 6 neurons are single-peaked. The 0.30 octaves is 3 envelope channels.

### Modulation, nonlinearity and feature selectivity for the neurons with all three STRFs

RTFs (D11), nonlinearities and their fits (the code's `fio_to_fio_fit.m`, D12) and FSI (the
code's `calculate_whole_cell_fsi.m`, D13) for the main group G, the 37 neurons both D7 readings
select; main reading on the significant STRF, the full STRF as the variant. Fixed seeds for the
fit starts and the FSI's random times, one per group.

How outcomes are graded. Each Table 1 claim is graded on the main group, under the readings its
criterion names; the two variant groups (the code's analytic significance, 29 neurons; the
cluster test, 15 neurons) are printed on one line each, and each conclusion says which of them
would give a different outcome. The cluster-test group was the main group of the first round.

In [ ]:
rng = np.random.default_rng(20200227)
fg = {n["file"] for n in G}
for n, r in zip(cc, by["cc_dmr"]):
    if n["file"] in fg:
        K.add_rtf_nl_fsi(n["strf"], envf, r["trig"], rng, ("spike", "large", "small"))
rng_an = np.random.default_rng(20200228)
fn = {n["file"] for n in GN}
for n, r in zip(cc_an, by["cc_dmr"]):
    if n["file"] in fn:
        K.add_rtf_nl_fsi(n["strf"], envf, r["trig"], rng_an, ("spike", "large", "small"))
for gl, gg in GROUPS:
    for n in gg:
        for c in ("spike", "large", "small"):
            n["strf"][c]["evrate"] = n["rate"][c]
PAIRS = (("spike", "large"), ("spike", "small"), ("large", "small"))
def col(key, c, nn=None):
    return np.array([n["strf"][c].get(key, np.nan) for n in (G if nn is None else nn)], float)
def paired(key, nn):
    """Means and paired t-tests of the three types in group nn."""
    v = {c: col(key, c, nn) for c in ("spike", "large", "small")}
    res = {}
    for a, b in PAIRS:
        ok = np.isfinite(v[a]) & np.isfinite(v[b])
        t = stats.ttest_rel(v[a][ok], v[b][ok]) if ok.sum() > 2 else None
        res[(a, b)] = (np.mean(v[b][ok] - v[a][ok]) if ok.any() else np.nan, t.statistic if t else np.nan, t.pvalue if t else np.nan, int(ok.sum()))
    return v, res
def tab(key, unit="", variants=True):
    """The main group in full, then one line per variant group (the code's analytic significance,
    the cluster test)."""
    out, res = paired(key, G)
    for c in ("spike", "large", "small"):
        v = out[c]
        print(f"  {c}: {np.nanmean(v):.2f} +- {np.nanstd(v):.2f} {unit} (n = {np.isfinite(v).sum()}, range {np.nanmin(v):.2f} to {np.nanmax(v):.2f})")
    for (a, b), (d, t, p, n) in res.items():
        print(f"  {a} vs {b}: mean difference {d:+.2f}, paired t = {t:.2f}, p = {p:.3g} (n = {n})")
    if variants:
        for gl, gg in GROUPS[1:]:
            va, rv = paired(key, gg)
            line = ", ".join(f"{c} {np.nanmean(va[c]):.2f}" for c in va)
            print(f"  variant {gl} (n = {len(gg)}): " + line + "; " + "; ".join(f"{a}-{b} {d:+.2f} p = {p:.3g}" for (a, b), (d, t, p, n) in rv.items()))
    return out
print("fits with nmse > 0.5:", int(sum((col('nl_nmse', c) > 0.5).sum() for c in ('spike', 'large', 'small'))), "of", 3 * len(G))
print("%.0f s so far" % (time.time() - T0))

### fig4c-btmf-small-psp-higher
> Best temporal modulation frequencies were 6.35 +- 4.50 Hz for spikes, 8.39 +- 4.69 Hz for large
> PSPs and 12.60 +- 4.26 Hz for small PSPs, significantly higher for small PSPs than for spikes
> and for large PSPs.

In [ ]:
for key, lab in (("btmf", "significant STRF, bandpass (a)"), ("btmf_b", "significant STRF, bandpass (b)"), ("btmf_full", "full STRF")):
    print(lab); o = tab(key, "Hz")
    d1 = np.nanmean(o["small"] - o["spike"]); d2 = np.nanmean(o["small"] - o["large"])
    vals = [within(np.nanmean(o["spike"]), 6.35, 4.5), within(np.nanmean(o["large"]), 8.39, 4.69), within(np.nanmean(o["small"]), 12.6, 4.26),
            within(d1, 6.25, 5.43), within(d2, 4.21, 5.34)]
    print("  values within:", vals)

**Result: not reproduced.** In the 37 neurons, under no reading of D11 are small PSPs above both spikes and large PSPs. Under reading (a) the bTMFs are 9.34, 10.51 and 11.18 Hz (spikes, large, small), within tolerance, with no difference (p = 0.341 against spikes, 0.643 against large PSPs). Under reading (b) small PSPs exceed spikes (15.21 against 11.96 Hz, p = 0.0212) but not large PSPs (p = 0.177), and the spike and large-PSP means fall outside tolerance. The full STRF shows no difference. Of the variants, the code's analytic group under (b) has small PSPs above spikes (p = 0.0149) but not above large PSPs (p = 0.107); only the 15-neuron cluster-test group under (b) meets the criterion (p = 0.004 and 0.0385), the outcome of the first round.

### fig4d-bsmf-small-psp-higher
> Best spectral modulation frequencies were 0.93 +- 0.68 cyc/oct for spikes, 0.78 +- 0.53 for
> large PSPs and 1.36 +- 0.70 for small PSPs, significantly different between small PSPs and
> spikes and between small and large PSPs.

In [ ]:
for key, lab in (("bsmf", "significant STRF, bandpass (a)"), ("bsmf_b", "significant STRF, bandpass (b)"), ("bsmf_full", "full STRF")):
    print(lab); o = tab(key, "cyc/oct")
    d1 = np.nanmean(o["small"] - o["spike"]); d2 = np.nanmean(o["small"] - o["large"])
    vals = [within(np.nanmean(o["spike"]), 0.93, 0.68), within(np.nanmean(o["large"]), 0.78, 0.53), within(np.nanmean(o["small"]), 1.36, 0.70),
            within(d1, 0.44, 0.89), within(d2, 0.58, 0.95)]
    print("  values within:", vals)

**Result: not reproduced.** The bSMF means fall within tolerance under every reading in the 37 neurons (reading (a): 0.88, 0.89 and 0.87 cyc/oct), but small PSPs are not higher than spikes or large PSPs under any reading (smallest p = 0.255). Neither variant group shows the difference (smallest p = 0.0765, in the cluster-test group).

### tab1-modulation-large-psp-like-spikes
> The modulation preferences of large non-spiking PSPs are similar to those of spikes, with no
> significant difference in bTMF or bSMF.

Judged on the paired t-tests of spikes against large PSPs printed in the two sections above
(main reading: significant STRF, bandpass (a)).

In [ ]:
for gl, gg in GROUPS:
  for key in ("btmf", "bsmf"):
    a, b = col(key, "spike", gg), col(key, "large", gg); ok = np.isfinite(a) & np.isfinite(b)
    t = stats.ttest_rel(a[ok], b[ok])
    print(f"{gl} {key}: spikes {np.mean(a[ok]):.2f}, large PSPs {np.mean(b[ok]):.2f}, difference {np.mean(b[ok] - a[ok]):+.2f}, p = {t.pvalue:.3g}")

**Result: reproduced.** In the 37 neurons, spikes and large PSPs do not differ in bTMF (9.34 against 10.51 Hz, p = 0.466) or in bSMF (0.88 against 0.89 cyc/oct, p = 0.978). Both variant groups agree (p = 0.218 and 0.252 in the 29; 0.845 and 0.987 in the 15).

### fig4b-bsmf-btmf-correlation
> Correlations between bSMF and bTMF were weak or moderate, significant only for large PSPs
> (spikes r = 0.28, large PSPs r = 0.53, small PSPs r = -0.19).

In [ ]:
for gl, gg in GROUPS:
    for c in ("spike", "large", "small"):
        x, y = col("btmf", c, gg), col("bsmf", c, gg); ok = np.isfinite(x) & np.isfinite(y)
        rr, p = stats.pearsonr(x[ok], y[ok])
        print(f"{gl} {c}: r = {rr:.2f}, p = {p:.3g}, n = {ok.sum()}")

**Result: not reproduced.** In the 37 neurons, the large-PSP correlation of bSMF with bTMF is r = 0.09 (p = 0.594), not 0.53; for spikes r = 0.18 and for small PSPs r = 0.04, none significant. The variant groups give r = -0.10 to 0.13, none significant.

### fig5a-cf-bf-correlation
> TRF characteristic frequencies and all-event STRF best frequencies were well correlated
> (r = 0.73, n = 40).

BF of the all-event STRF, the band closest to CF for two bands (Results); CF from the TRF.
Readings: kHz and log2 frequency. Neurons without a significant all-event subfield have no BF
and are counted.

In [ ]:
x = np.array([cfs[n["file"]] for n in cc]); y = np.array([n["strf"]["all"].get("bf", np.nan) for n in cc])
ok = np.isfinite(x) & np.isfinite(y)
print("neurons with a BF:", ok.sum(), "of", len(cc))
for lab, fx in (("kHz", lambda v: v / 1000), ("log2", np.log2)):
    rr, p = stats.pearsonr(fx(x[ok]), fx(y[ok]))
    print(f"{lab}: r = {rr:.2f}, p = {p:.2g}; within 0.73 +- 0.073: {within(rr, 0.73, 0.073)}")
print("median |log2(BF/CF)| = %.2f octaves; within 0.5 octave: %d" % (np.median(np.abs(np.log2(y[ok] / x[ok]))), (np.abs(np.log2(y[ok] / x[ok])) < 0.5).sum()))
xs = np.array([cfs[n["file"]] for n in cc_an]); ys = np.array([n["strf"]["all"].get("bf", np.nan) for n in cc_an]); ok2 = np.isfinite(ys)
print("variant, analytic significance (code): n = %d, r(log2) = %.2f" % (ok2.sum(), stats.pearsonr(np.log2(xs[ok2]), np.log2(ys[ok2]))[0]))
# Diagnosis. (1) The TRF of one neuron, peak response (mV) per frequency and level, with its CF.
tr0 = next(r for r in tones if r["file"] == cc_pairs[cc[0]["file"]]); t0_ = tr0["t"]
fr_u, lv_u = np.unique(t0_["freq"]), np.unique(t0_["level"])
M = np.full((len(lv_u), len(fr_u)), np.nan)
for k in range(len(t0_["freq"])):
    M[np.searchsorted(lv_u, t0_["level"][k]), np.searchsorted(fr_u, t0_["freq"][k])] = t0_["post"][k].max() if t0_["resp"][k] else 0
print(f"TRF of tone recording {tr0['file']} (rows: dB SPL from 70 down; columns: every 4th of the 45 frequencies; 0 = no response), CF {tr0['cf_aa'] / 1000:.1f} kHz")
print("  kHz  " + " ".join(f"{f / 1000:5.1f}" for f in fr_u[::4]))
for j in range(len(lv_u) - 1, -1, -1):
    print(f"  {lv_u[j]:3.0f}  " + " ".join(f"{v:5.1f}" for v in M[j, ::4]))
# (2) CF against the TRF's best frequency (largest summed response over all levels, and over 40 to 70 dB SPL)
def trf_bf(t, lmin):
    resp = t["post"].max(1) * t["resp"]; fr = np.unique(t["freq"])
    return fr[np.argmax([resp[(t["freq"] == f) & (t["level"] >= lmin)].sum() for f in fr])]
tb_all = np.array([trf_bf(next(r for r in tones if r["file"] == cc_pairs[n["file"]])["t"], 0) for n in cc])
tb_loud = np.array([trf_bf(next(r for r in tones if r["file"] == cc_pairs[n["file"]])["t"], 40) for n in cc])
print("CF vs TRF best frequency at 40-70 dB SPL: r(log2) = %.2f (p = %.2g); CF vs TRF best frequency over all levels: r = %.2f"
      % (*stats.pearsonr(np.log2(x), np.log2(tb_loud)), stats.pearsonr(np.log2(x), np.log2(tb_all))[0]))
for lab, ref in (("TRF best frequency, all levels", tb_all), ("TRF best frequency, 40-70 dB SPL", tb_loud)):
    rr, p = stats.pearsonr(np.log2(ref[ok]), np.log2(y[ok]))
    print(f"variant, STRF BF against the {lab}: r(log2) = {rr:.2f}, p = {p:.2g}, n = {ok.sum()}")
print("responsive tones at the 0 dB SPL threshold level per neuron: median %d of 45" % np.median([np.sum(tr["t"]["resp"] & (tr["t"]["level"] == tr["thr_aa"])) for tr in tones]))
# (3) where the STRF BFs fall on the envelope's 64 channels
bfc = collections.Counter(np.round(y[ok] / 1000, 1).tolist())
print("all-event STRF BFs (kHz) by count:", bfc.most_common(6), "| distinct channels:", len(bfc))

**Result: not reproduced.** In the 35 of 40 neurons whose all-event STRF has a subfield, the correlation between TRF CF and STRF BF is r = 0.04 on kHz and -0.01 on log2 frequency, and the median BF lies 0.51 octaves from the CF; the code's analytic significance (39 neurons) gives r = 0.23. The diagnosis points at the CF: it is set at the threshold level (0 dB SPL in 64 of 66 neurons), where a median of 5 of 45 tones per neuron pass the response criterion on a single presentation, and it does not correlate with the TRF's best frequency at 40 to 70 dB SPL (r = -0.02). The example TRF printed shows scattered single responses, not a tuning curve. Against the TRF's best frequency over all levels, the STRF BF correlates at r = 0.37 (p = 0.027), still far from 0.73. The STRF BFs fall on only 9 envelope channels: 11 at 20.0 kHz, 8 at 18.6 kHz and 7 at 12.3 kHz.

### fig5b-bf-same-across-events
> Best frequencies of spike STRFs closely matched those of large and small PSP STRFs, with no
> between-group difference.

In [ ]:
for gl, gg in GROUPS:
    B = {c: col("bf", c, gg) for c in ("spike", "large", "small")}
    ok = np.all([np.isfinite(B[c]) for c in B], axis=0)
    for lab, fx in (("kHz", lambda v: v / 1000), ("log2", np.log2)):
        a = stats.f_oneway(*[fx(B[c][ok]) for c in B])
        print(f"{gl} (n = {ok.sum()} with a BF for all three): one-way ANOVA ({lab}): F = {a.statistic:.2f}, p = {a.pvalue:.3g}")
    for c in ("large", "small"):
        d = np.abs(np.log2(B[c][ok] / B["spike"][ok]))
        print(f"  spike vs {c}: median |difference| {np.median(d):.2f} octaves, within 0.5 octave {np.sum(d < 0.5)} of {len(d)}")

**Result: reproduced.** In the 15 of the 37 neurons with a BF in all three STRFs (a BF needs a subfield kept by the cluster test), the one-way ANOVA of BF across spikes, large and small PSPs gives p = 0.682 (kHz) and 0.46 (log2), and the median spike-PSP differences are 0.10 (large) and 0.00 (small) octaves. In the code's analytic group (29 neurons with a BF) the ANOVA gives p = 0.504 and 0.628, with median differences of 0.90 and 0.70 octaves.

### fig5c-bandwidth-relationships
> Spike STRFs with a single band were commonly sharply tuned below about 0.5 octaves (20/30); in
> 13/20 (large PSPs) and 15/20 (small PSPs) of these the PSP bandwidths were also below 0.5
> octaves; 9/30 (large) and 6/30 (small) PSP STRFs had total bandwidths above 0.5 octaves largely
> from secondary peaks; 8/30 spike STRFs wider than 0.6 octaves had PSP STRFs of similar
> bandwidth; one double-peaked spike STRF had single-peaked PSP STRFs below 0.5 octaves.

In [ ]:
# counts as proportions of the group, so that a group of n neurons is compared with the paper's 30
# (the "of these" counts as proportions of the narrow single-band spike STRFs, the paper's 20)
for gl, gg in GROUPS:
  for bwk in ("total_a", "total_b"):
    S = col(bwk, "spike", gg); Lg = col(bwk, "large", gg); Sm = col(bwk, "small", gg); nb = col("nbands", "spike", gg)
    n_ = int(np.isfinite(S).sum())
    narrow = (nb == 1) & (S < 0.5)
    c1 = int(narrow.sum()); c2 = int((narrow & (Lg < 0.5)).sum()); c3 = int((narrow & (Sm < 0.5)).sum())
    c4a = int((narrow & (Lg > 0.5)).sum()); c5a = int((narrow & (Sm > 0.5)).sum())
    c4b = int(((S < 0.6) & (Lg > 0.5)).sum()); c5b = int(((S < 0.6) & (Sm > 0.5)).sum())
    c6 = int((S > 0.6).sum())
    c7 = int(((nb == 2) & (col("nbands", "large", gg) == 1) & (col("nbands", "small", gg) == 1) & (Lg < 0.5) & (Sm < 0.5)).sum())
    def prop(x, d):
        return x / d if d else np.nan
    for rd, (x4, x5, d45, e45) in (("(a)", (c4a, c5a, c1, 20)), ("(b)", (c4b, c5b, n_, 30))):
        got = [(prop(c1, n_), 20 / 30), (prop(c2, c1), 13 / 20), (prop(c3, c1), 15 / 20), (prop(x4, d45), 9 / e45), (prop(x5, d45), 6 / e45), (prop(c6, n_), 8 / 30)]
        hits = sum(within(g, e, e / 10) for g, e in got)
        print(f"{gl}, {bwk}, reading {rd} for (ii): n = {n_}; narrow single-band spike {c1}; of these large < 0.5 {c2}, small < 0.5 {c3}; "
              f"(ii) large > 0.5 {x4}, small > 0.5 {x5}; spike > 0.6 {c6}; (iv) {c7}")
        print("    proportions " + ", ".join(f"{g:.2f} (paper {e:.2f})" for g, e in got) + f"; within 10% of the paper's proportion: {hits} of 6")

**Result: not reproduced.** The counts are compared as proportions of the group, since no group here has 30 neurons. In the 37 neurons (28 with a spike band), with D8 (a), 8 single-band spike STRFs are below 0.5 octaves (0.29 of the 28, against 20/30 = 0.67), and of these 5 and 3 have large- and small-PSP STRFs below 0.5 octaves (0.62 and 0.38, against 0.65 and 0.75). One of the six proportions falls within 10% of the paper's under each bandwidth and reading, and none or one in the variant groups.

### tab1-single-band-small-psp-narrower
> Individual-peak STRF bandwidths were 0.32 +- 0.12 octaves for spikes, 0.28 +- 0.11 for large
> PSPs and 0.25 +- 0.07 for small PSPs; spike and large-PSP bandwidths did not differ, small PSPs
> were slightly narrower (about 20%), significantly so against spikes.

D8 readings for the bands: (a) every band that can be matched across the three types (here:
neurons where all three STRFs have the same number of bands, bands paired in frequency order),
(b) one band per neuron, the main band (closest to CF).

In [ ]:
print("reading (b), main band per neuron:"); ob = tab("single_main", "oct")
pa = {c: [] for c in ("spike", "large", "small")}
for n in G:
    nbs = [len(n["strf"][c].get("single_a", [])) for c in pa]
    if len(set(nbs)) == 1 and nbs[0] > 0:
        for c in pa:
            bands = sorted(zip(n["strf"][c]["peak_freqs"], n["strf"][c]["single_a"]))
            pa[c] += [b for _, b in bands]
pa = {c: np.array(v) for c, v in pa.items()}
print("reading (a), matched bands: n =", len(pa["spike"]))
for c in pa:
    print(f"  {c}: {pa[c].mean():.2f} +- {pa[c].std():.2f}")
for a, b in (("spike", "large"), ("spike", "small"), ("large", "small")):
    t = stats.ttest_rel(pa[a], pa[b]); print(f"  {a} vs {b}: p = {t.pvalue:.3g}")
print("one-way ANOVA, reading (b): p = %.3g" % stats.f_oneway(*[ob[c][np.isfinite(ob[c])] for c in ob]).pvalue)

**Result: partially reproduced.** With one main band per neuron (reading (b)), the bandwidths in the 37 neurons are 0.44, 0.46 and 0.34 octaves (spikes, large, small; n = 28, 22 and 30). Small PSPs are the narrowest but not significantly narrower than spikes (p = 0.184), which the criterion grades partial; spikes and large PSPs do not differ (p = 0.221). The large and small means fall outside tolerance (0.28 +- 0.11, 0.25 +- 0.07). Only 2 neurons have bands that can be matched across the three types (reading (a)). In the code's analytic group small PSPs are not the narrowest (0.17, 0.21 and 0.19).

### tab1-total-bandwidth-no-difference
> Total STRF bandwidths (all bands) did not differ between spikes (0.50 +- 0.32 octaves), large
> PSPs (0.57 +- 0.33) and small PSPs (0.50 +- 0.32).

In [ ]:
print("D8 (a) total bandwidth:"); oa = tab("total_a", "oct")
print("D8 (b) half-maximum width:"); _ = tab("total_b", "oct")
print("variant, main band only (no second band):"); _ = tab("single_main", "oct", variants=False)

**Result: not reproduced.** In the 37 neurons, small PSPs have narrower total bandwidths than spikes under both D8 readings: 0.77 against 1.52 octaves with D8 (a) (p = 0.0203, 22 pairs) and 0.31 against 0.54 with the half-maximum width D8 (b) (p = 0.00569). The D8 (a) means of spikes and large PSPs (1.52 and 1.06 octaves) lie above their tolerances, raised by wide second bands; the main band alone gives 0.44, 0.46 and 0.34. In the code's analytic group no pair differs under D8 (a) (p >= 0.129); in the 15-neuron cluster-test group no pair differs under D8 (a), the outcome of the first round, but two do under D8 (b) (p = 0.0433 and 0.0266).

### tab1-latency-duration
> STRF latencies were 42.7 +- 11.1 ms (spikes), 48.7 +- 11.8 ms (large PSPs) and 45.2 +- 8.2 ms
> (small PSPs), large PSPs significantly later than spikes and small PSPs; durations were 43.0 +-
> 24.1, 43.6 +- 18.8 and 34.5 +- 10.3 ms, small PSPs significantly shorter than spikes and large PSPs.

In [ ]:
for rd in "ab":
    print(f"latency, D10 ({rd}):"); _ = tab(f"lat_{rd}", "ms")
    print(f"duration, D10 ({rd}):"); _ = tab(f"dur_{rd}", "ms")

**Result: not reproduced.** In the 37 neurons, under peak latency (D10 (a)) the latencies are 54.3, 64.3 and 55.3 ms (spikes, large, small), the spike and large-PSP means outside tolerance. Under the earliest-pixel latency (b) they are 34.3, 48.6 and 43.3 ms, all within tolerance, and the durations under (a) (32.7, 34.3 and 26.8 ms) are within as well. But of Table 1's four significant pairs only one then holds (duration, large against small PSPs, p = 0.0448); latency spikes against large PSPs gives p = 0.159, large against small p = 0.0863, and duration spikes against small p = 0.261. The criterion's partial grade needs two or three of the four.

### tab1-event-rate-no-difference
> STRF event rates did not differ between spikes (1.31 +- 0.92 Hz), large PSPs (1.73 +- 0.72 Hz)
> and small PSPs (1.52 +- 0.56 Hz).

Event rate = events of the type over the analysed stimulus time (blocks 2 to N-2 of the DMR,
about 600 s), the claims file's standard choice.

In [ ]:
_ = tab("evrate", "Hz")

**Result: not reproduced.** In the 37 neurons, large PSPs occur at a higher rate than small PSPs (0.90 against 0.76 Hz, p = 0.00804); spikes (1.09 Hz) differ from neither (p = 0.288 and 0.0734). The large and small means lie below their tolerances (1.73 +- 0.72, 1.52 +- 0.56). The code's analytic group also has large above small (p = 0.0378); the 15-neuron cluster-test group shows no difference (smallest p = 0.0623), the outcome of the first round.

### tab1-fsi-spikes-higher-than-large
> Feature selectivity index was 0.15 +- 0.08 for spikes, 0.12 +- 0.03 for large PSPs and 0.15 +-
> 0.04 for small PSPs; spikes and small PSPs were significantly higher than large PSPs, and spikes
> did not differ from small PSPs.

In [ ]:
_ = tab("fsi")

**Result: reproduced.** In the 37 neurons, the FSI means are 0.19, 0.11 and 0.13 (spikes, large, small), all within tolerance. Spikes exceed large PSPs (p = 0.0302), small PSPs exceed large PSPs (p = 0.00524), and spikes and small PSPs do not differ (p = 0.11). Neither variant group meets the criterion: spikes do not exceed large PSPs in the 29 (p = 0.24) or in the 15 (p = 0.395).

### fig6b-tab1-threshold-spikes-higher
> Nonlinearity thresholds were significantly higher for spikes (0.61 +- 0.87 s.d.) than for large
> (-0.57 +- 0.87) and small PSPs (-0.55 +- 0.74), which did not differ.

The fit form is the code's (`ringach_malone_func.m`, `fio_to_fio_fit.m`), so D12's form is
settled by the paper's code; threshold = theta.

In [ ]:
print("significant STRF (D12 (a)):"); th = tab("theta", "s.d.")
print("full STRF (D12 (b)):"); thf = tab("theta_full", "s.d.")

**Result: not reproduced.** In the 37 neurons, with the code's fit on the significant STRF (D12 (a)), the mean thresholds are 2.12, 0.84 and 1.12 s.d. (spikes, large, small), all outside tolerance; spikes exceed large PSPs (p = 0.0328) but not small ones (p = 0.0571). The full STRF (D12 (b)) gives no significant difference (p = 0.096 and 0.0515). The code's analytic group has spikes above large PSPs (p = 0.00417) but not above small ones (p = 0.27). 20 of the 111 fits have a normalized m.s.e. above 0.5.

### fig6c-tab1-transition-spikes-lower
> Nonlinearity transitions were smaller for spikes (1.25 +- 0.48 s.d.) than for large (1.93 +-
> 0.67) and small PSPs (1.59 +- 0.27), and smaller for small than for large PSPs.

In [ ]:
print("significant STRF (D12 (a)):"); trn = tab("trans", "s.d.")
print("full STRF (D12 (b)):"); trf = tab("trans_full", "s.d.")
print("negative transitions (the fit is unconstrained, as in the code):", int(sum((trn[c] < 0).sum() for c in trn)))

**Result: not reproduced.** In the 37 neurons the mean transitions are 0.82, 0.84 and 0.87 s.d. (spikes, large, small), with no difference (p = 0.823, 0.578 and 0.722); the large- and small-PSP means lie below their tolerances. The full STRF and the variant groups show no difference either (all p >= 0.305).

### fig6bc-no-correlation-psp-spike-nonlinearity
> Neither nonlinearity thresholds nor transitions were correlated between PSPs and spikes of the
> same neurons.

In [ ]:
for gl, gg in GROUPS:
    for key in ("theta", "trans"):
        for c in ("small", "large"):
            x, y = col(key, c, gg), col(key, "spike", gg); ok = np.isfinite(x) & np.isfinite(y)
            rr, p = stats.pearsonr(x[ok], y[ok]); print(f"{gl} {key}, {c} vs spikes: r = {rr:.3f}, p = {p:.3g}, n = {ok.sum()}")

**Result: reproduced.** In the 37 neurons none of the four correlations between PSP and spike nonlinearity parameters is significant: thresholds r = 0.159 (small PSPs) and -0.078 (large), transitions r = 0.216 and 0.306, the smallest p 0.0653. The code's analytic group agrees (|r| <= 0.08). In the 15-neuron cluster-test group the small-PSP and spike transitions correlate (r = 0.539, p = 0.0383), the outcome of the first round.

### fig6d-spike-threshold-transition-covary
> For spikes, nonlinearity threshold and transition covaried negatively (r = -0.64), with only
> weak, non-significant correlations for large or small PSPs.

In [ ]:
for gl, gg in GROUPS:
    for c in ("spike", "large", "small"):
        x, y = col("theta", c, gg), col("trans", c, gg); ok = np.isfinite(x) & np.isfinite(y)
        rr, p = stats.pearsonr(x[ok], y[ok]); print(f"{gl} {c}: r = {rr:.2f}, p = {p:.3g}, n = {ok.sum()}")

**Result: partially reproduced.** In the 37 neurons the spike threshold and transition correlate negatively and significantly (r = -0.37, p = 0.0248), outside -0.64 +- 0.064, and neither PSP correlation is significant (large r = -0.09, p = 0.61; small r = 0.28, p = 0.0963), which the criterion grades partial. In the code's analytic group spikes give r = -0.41 (p = 0.0286), but small PSPs also correlate (r = -0.64, p = 0.000196); in the cluster-test group spikes give r = -0.02.

### fig6efg-fsi-relationships
> Feature selectivity index rose with nonlinearity threshold and fell with transition and with
> event rate, clearly for spikes (r = 0.68, -0.67, -0.63) and more weakly for PSPs.

D17 for the power fits: (a) r on raw values, (b) r on log values (pairs with non-positive values
dropped).

In [ ]:
for gl, gg in GROUPS:
  for c in ("spike", "large", "small"):
    f_ = col("fsi", c, gg)
    for key in ("theta", "trans", "evrate"):
        x = col(key, c, gg); ok = np.isfinite(x) & np.isfinite(f_)
        r1, p1 = stats.pearsonr(x[ok], f_[ok])
        pos = ok & (x > 0) & (f_ > 0)
        r2, p2 = stats.pearsonr(np.log(x[pos]), np.log(f_[pos])) if pos.sum() > 2 else (np.nan, np.nan)
        print(f"{gl} {c}, {key} vs FSI: raw r = {r1:.2f} (p = {p1:.3g}); log r = {r2:.2f} (p = {p2:.3g}, n = {pos.sum()})")

**Result: not reproduced.** In the 37 neurons the three spike correlations have the expected signs with p < 0.001 (raw r = 0.82 for threshold, -0.58 for transition and -0.54 for event rate), but all three lie outside tolerance, where the partial grade allows one; on log values they are 0.53, -0.79 and -0.98, also outside. The PSP event-rate correlations (raw r = -0.91 and -0.84) are stronger than the spike one. The code's analytic group would be graded partial: spike r = 0.38 (p = 0.0407), -0.63 and -0.61, the last two within tolerance.

### fig5df-strf-narrower-than-trf
> STRF bandwidths were substantially narrower than TRF bandwidths; PSP-based TRF bandwidth30s
> exceeded total STRF bandwidths by about 1 octave for all three event types (0.99 +- 0.45, 0.92
> +- 0.44 and 0.99 +- 0.44 octaves for spikes, large and small PSPs), with no group difference.

In [ ]:
for gl, gg in GROUPS:
  for trf_key in ("bw_psp_aa", "bw_psp_ab"):
    trfbw = np.array([next(r[trf_key] for r in tones if r["file"] == cc_pairs[n["file"]]) for n in gg])
    D = {c: trfbw - col("total_a", c, gg) for c in ("spike", "large", "small")}
    print(f"{gl} (n = {len(gg)}), TRF {trf_key}: mean {np.nanmean(trfbw):.2f} oct; STRF total bandwidth " +
          ", ".join(f"{c} {np.nanmean(col('total_a', c, gg)):.2f}" for c in D))
    for c, e, t, me in (("spike", 0.99, 0.45, 1.02), ("large", 0.92, 0.44, 1.01), ("small", 0.99, 0.44, 0.91)):
        v = D[c]
        print(f"  {c}: delta {np.nanmean(v):.2f} +- {np.nanstd(v):.2f} (n = {np.isfinite(v).sum()}, within {within(np.nanmean(v), e, t)}), median {np.nanmedian(v):.2f} (within {within(np.nanmedian(v), me, me / 10)})")
    print("  one-way ANOVA p = %.3g" % stats.f_oneway(*[D[c][np.isfinite(D[c])] for c in D]).pvalue)

**Result: not reproduced.** In the 37 neurons, the PSP TRF bandwidth30 (2.11 octaves on average) exceeds the total STRF bandwidth by 0.46, 1.14 and 1.30 octaves (spikes, large, small; n = 28, 22 and 30). The spike difference falls outside 0.99 +- 0.45, and the one-way ANOVA gives p = 0.0692; the medians (0.51, 1.61 and 1.55) are all outside 10%. The contiguous TRF reading (D15 (b)) makes the differences negative. The code's analytic group gives 0.10, -0.18 and 0.55. Only the 15-neuron cluster-test group meets the criterion (0.57, 0.91 and 1.28, ANOVA p = 0.397), the outcome of the first round, with TRF and STRF widths (1.97; 0.69 to 1.40 octaves) both larger than the paper's.

### fig5e-q-strf-higher
> Q factors of all-event STRFs were significantly higher than TRF Q30 values for PSP TRFs (n = 40)
> and for spike TRFs (n = 15 with NDR of at least 0.3).

STRF Q = BF / all-event total bandwidth; TRF Q30 readings (a) CF / bandwidth30, (b) frequency of
the largest response at threshold + 30 dB / bandwidth30; bandwidth in kHz (D15 (a)) or octaves (b).

In [ ]:
trow = {cc_pairs[n["file"]]: next(r for r in tones if r["file"] == cc_pairs[n["file"]]) for n in cc}
print("paired neurons (tones and DMR) with NDR >= 0.3:", sum(trow[k]["ndr_aa"] >= 0.3 for k in trow), "of", len(trow),
      "; with a spike bandwidth30 and an all-event STRF BF, used in the spike test, as printed below")
for unit in ("khz", "oct"):
    for qr in "ab":
        sq, pq, sq2, kq = [], [], [], []
        for n in cc:
            m = n["strf"]["all"]; tr_ = trow[cc_pairs[n["file"]]]
            if "bf" not in m:
                continue
            bw_s = m["total_a_khz"] if unit == "khz" else m["total_a"]
            if qr == "a":
                fref = tr_["cf_aa"]
            else:
                t = tr_["t"]; lv = tr_["thr_aa"] + 30; msk = t["resp"] & (t["level"] == lv)
                fref = t["freq"][msk][np.argmax(t["post"][msk].max(1))] if msk.any() else np.nan
            bwp = tr_["bw_psp_khz_aa"] if unit == "khz" else tr_["bw_psp_aa"]
            bws = tr_["bw_spk_khz_aa"] if unit == "khz" else tr_["bw_spk_aa"]
            q_s = m["bf"] / 1000 / bw_s if unit == "khz" else m["bf"] / 1000 / bw_s
            fr_ = fref / 1000
            if np.isfinite(bwp) and bwp > 0:
                sq.append(q_s); pq.append(fr_ / bwp)
            if tr_["ndr_aa"] >= 0.3 and np.isfinite(bws) and bws > 0:
                sq2.append(q_s); kq.append(fr_ / bws)
        t1 = stats.ttest_rel(sq, pq); t2 = stats.ttest_rel(sq2, kq) if len(sq2) > 2 else None
        print(f"bandwidth in {unit}, Q30 reading ({qr}): PSP n = {len(sq)}, STRF Q {np.mean(sq):.2f} vs TRF Q30 {np.mean(pq):.2f}, p = {t1.pvalue:.2g}; "
              f"spike n = {len(sq2)}" + (f", STRF Q {np.mean(sq2):.2f} vs {np.mean(kq):.2f}, p = {t2.pvalue:.2g}" if t2 else ""))

**Result: reproduced.** Of the 40 paired neurons, 15 have NDR of at least 0.3. All-event STRF Q is higher than PSP TRF Q30 (2.88 against 0.87 with bandwidths in kHz, n = 35, p = 2.4e-06), and higher than spike TRF Q30 in the 11 neurons with a spike bandwidth30 (3.39 against 1.37, p = 0.028). Every Q30 and bandwidth reading agrees (spike p = 0.019 to 0.028). The STRF bandwidth in kHz now spans the same channels as the width in octaves.

## DMR, voltage clamp (Fig. 7, Table 2)

Events: peakfinder with selectivity 20 pA and 40 pA, Methods "~20-40 pA", on the inverted current
for excitation and the current for inhibition; kept above 3 x the baseline s.d.; one STRF over
all kept events per recording (Results). D3 readings for the baseline s.d.: (b) the noise of
event-free 50-ms segments during the DMR, and (a) the s.d. of the pre-stimulus current. Three event
sets are carried through every claim: 20 pA with (b), 40 pA with (b), and 20 pA with (a); 40 pA
with (a) is run for the event counts. Current-clamp comparisons use the main group G.

In [ ]:
rng_vc = np.random.default_rng(7)
def vc_table(sel, with_nl=True, sd="noise"):
    rows = []
    for r in by["vc_exc"] + by["vc_inh"]:
        keep, a, s = K.vc_events(r, sel=sel, sd=sd)
        ev = {"all": keep}
        st = K.analyse_strfs(r, envf, faxis, ev, classes=("all",), key=f"pk{sel}")
        if with_nl:
            K.add_rtf_nl_fsi(st, envf, r["trig"], rng_vc, ("all",))
        m = st["all"]
        m["evrate"] = m["n0"] / ((len(r["trig"]) - 3) * K.NT * K.DT_ENV)
        rows.append(dict(file=r["file"], kind=r["kind"], m=m, amp=a[keep], sd=s, nev=int(keep.sum())))
    return rows
vc = vc_table(20)
vc40 = vc_table(40)
vc_pre = vc_table(20, sd="pre")                     # D3 reading (a): s.d. of the pre-stimulus current
vc40_pre = vc_table(40, with_nl=False, sd="pre")
VCSETS = ((vc, "20 pA"), (vc40, "40 pA"), (vc_pre, "20 pA, pre-stimulus s.d."))
for rows, lab in ((vc, "20 pA, noise s.d."), (vc40, "40 pA, noise s.d."), (vc_pre, "20 pA, pre-stimulus s.d."), (vc40_pre, "40 pA, pre-stimulus s.d.")):
    for k in ("vc_exc", "vc_inh"):
        rr = [x for x in rows if x["kind"] == k]
        print(f"{lab} {k}: events kept per recording {min(x['nev'] for x in rr)} to {max(x['nev'] for x in rr)}, "
              f"baseline s.d. {min(x['sd'] for x in rr):.1f} to {max(x['sd'] for x in rr):.1f} pA, significant subfield {sum(x['m']['sig_sub'] for x in rr)} of {len(rr)}")
def vcol(key, kind, rows=None):
    return np.array([x["m"].get(key, np.nan) for x in (vc if rows is None else rows) if x["kind"] == kind], float)
print("%.0f s so far" % (time.time() - T0))

### fig7-current-amplitude-unimodal
> Peak current amplitude distributions (events above 3 x s.d. of baseline) were unimodal for EPSCs
> and IPSCs.

In [ ]:
for rows, lab in VCSETS + ((vc40_pre, "40 pA, pre-stimulus s.d."),):
    uk = sum(K.bimodal_trough(x["amp"]) is None for x in rows if len(x["amp"]) >= 10)
    ud = sum(K.dip_test_p(x["amp"]) >= 0.05 for x in rows if len(x["amp"]) >= 10)
    n = sum(len(x["amp"]) >= 10 for x in rows)
    print(f"{lab}: unimodal by KDE (D4 (a)) {uk} of {n} ({uk / n:.1%}); by dip test (D4 (b)) {ud} of {n} ({ud / n:.1%})")

**Result: partially reproduced.** By the dip test (D4 (b)), 97.4 to 100% of the 39 current recordings are unimodal under every selectivity and baseline reading. By the KDE reading (D4 (a)), 87.2% are at 20 pA, 71.8% at 40 pA, 89.7% at 20 pA with the pre-stimulus s.d. and 79.5% at 40 pA with it. The criterion asks for 90% under the readings; the KDE reading falls short under all four.

### fig7c-ei-bf-matched
> In the eight neurons with both currents, excitatory and inhibitory STRF best frequencies were
> closely matched.

In [ ]:
for rows, sl in VCSETS:
    bf = {x["file"]: x["m"].get("bf", np.nan) for x in rows}
    e = np.array([bf[a] for a, b in vc_pairs]); i_ = np.array([bf[b] for a, b in vc_pairs])
    print(sl, "pairs (kHz):", [(round(float(u) / 1000, 1), round(float(v) / 1000, 1)) for u, v in zip(e, i_)])
    ok = np.isfinite(e) & np.isfinite(i_)
    if ok.sum() > 2:
        rr, p = stats.pearsonr(np.log2(e[ok]), np.log2(i_[ok]))
        d = np.median(np.abs(np.log2(e[ok] / i_[ok])))
        print(f"  n = {ok.sum()}: Pearson r (log2) = {rr:.2f}, p = {p:.3g}; median |E - I| = {d:.2f} octaves; "
              f"reading (a) {rr > 0 and p < 0.05}, reading (b) {d < 0.32}")

**Result: partially reproduced.** Six of the eight pairs have a subfield in both STRFs. The median E-I best-frequency difference is 0.15 octaves under all three event sets, below 0.32 (reading (b) holds). The correlation is not significant under any (reading (a) fails): r = 0.74 (p = 0.096) at 20 pA, 0.53 (p = 0.28) at 40 pA and 0.38 (p = 0.452) with the pre-stimulus s.d. Five of the eight excitatory BFs at 20 pA lie on one envelope channel, 20.0 kHz.

### fig7d-current-double-peaks
> Double-peaked STRFs were seen in 35% (11/31) of the current recordings (9/27 excitatory, 5/12
> inhibitory), similar to small PSPs and about twice the rate for spikes.

In [ ]:
for rows, sl in VCSETS:
  for rd in "ab":
    de = {x["file"]: (x["m"].get("nbands", 0) == 2 if rd == "a" else bool(x["m"].get("double_b"))) for x in rows}
    ne_ = sum(de[f] for f in files["vc_exc"]); ni = sum(de[f] for f in files["vc_inh"])
    neur = 0
    for row in vc_rows:
        fs_ = [str(v) for v in (row[1], row[3]) if v]
        neur += any(de[f] for f in fs_)
    print(f"{sl}, D9 ({rd}): excitatory {ne_}/27, inhibitory {ni}/12, neurons {neur}/31, traces {ne_ + ni}/39; "
          f"within (11, 9, 5) reading (a) neurons: {within(neur, 11, 1.1) and within(ne_, 9, 0.9) and within(ni, 5, 0.5)}")

**Result: not reproduced.** With bands as separate subfields (D9 (a)), 13 of 27 excitatory and 2 of 12 inhibitory STRFs are double-peaked at 20 pA (15 of 31 neurons); 12, 4 and 15 at 40 pA; 15, 1 and 16 with the pre-stimulus s.d. The profile reading (b) gives 13 to 15, 10 to 12 and 20 to 23. None matches 9, 5 and 11.

### fig7e-tab2-inhibition-broader
> Inhibitory STRF bandwidths (0.89 +- 0.34 octaves) exceeded excitatory ones (0.65 +- 0.30) by
> about 27%, and exceeded current-clamp all-event bandwidths (0.64 +- 0.37), while excitatory and
> current-clamp bandwidths did not differ.

In [ ]:
def ut(a, b):
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    t = stats.ttest_ind(a, b); return f"{a.mean():.2f} +- {a.std():.2f} (n={len(a)}) vs {b.mean():.2f} +- {b.std():.2f} (n={len(b)}), p = {t.pvalue:.3g}"
ccall = np.array([n["strf"]["all"].get("total_a", np.nan) for n in cc])
for (rows, sl), bwk in [(s, b) for s in VCSETS for b in ("total_a", "total_b")]:
    print(sl, end=" ")
    E_, I_ = vcol(bwk, "vc_exc", rows), vcol(bwk, "vc_inh", rows)
    ccb = np.array([n["strf"]["all"].get(bwk, np.nan) for n in cc])
    print(bwk, "| E vs I:", ut(E_, I_)); print("   E vs CC all events:", ut(E_, ccb)); print("   I vs CC all events:", ut(I_, ccb))

**Result: reproduced, with the pre-stimulus s.d. (D3 (a)) at 20 pA and the half-maximum width (D8 (b)).** There the inhibitory STRFs (0.72 octaves, n = 9) are broader than the excitatory ones (0.41, n = 27; p = 0.0299) and than the current-clamp all-event STRFs (0.35, n = 35; p = 0.00429), and the three means are within tolerance; excitation and current clamp do not differ (p = 0.421). With the noise s.d. at 20 pA the effect holds (0.77 against 0.34 octaves, p = 0.000123), but the excitatory mean lies just below its tolerance (0.65 +- 0.30). At 40 pA (p = 0.634), and with the total bandwidth D8 (a) under every event set (p >= 0.386), inhibition is not broader. The match holds in one of the six combinations of event set and bandwidth.

### fig7f-tab2-ipsc-bsmf-lower
> IPSC STRFs had lower bSMFs (0.25 +- 0.21 cyc/oct) than EPSC STRFs (0.62 +- 0.47) and than spikes,
> large and small PSPs; EPSC bSMFs were lower than small PSPs.

In [ ]:
for key, lab in (("bsmf", "significant STRF, bandpass (a)"), ("bsmf_b", "significant STRF, bandpass (b)"), ("bsmf_full", "full STRF, bandpass (a)")):
    for rows, sl in VCSETS:
        E_, I_ = vcol(key, "vc_exc", rows), vcol(key, "vc_inh", rows)
        print(f"{lab}, {sl}: E vs I:", ut(E_, I_))
        for c in ("spike", "large", "small"):
            print(f"   E vs {c}:", ut(E_, col(key, c))); print(f"   I vs {c}:", ut(I_, col(key, c)))

**Result: not reproduced.** Under no reading or event set are IPSC bSMFs lower than EPSC bSMFs (IPSC 0.50 to 0.76 against EPSC 0.33 to 0.52 cyc/oct, E-I p >= 0.0582) or than spike bSMFs (p >= 0.22). The EPSC means lie within tolerance (0.62 +- 0.47); the IPSC means do not (0.25 +- 0.21).

### tab2-btmf-currents-higher-than-spikes
> Temporal modulation tuning of EPSCs (11.2 +- 6.0 Hz) and IPSCs (12.0 +- 6.9 Hz) was closely
> matched and significantly higher than bTMFs for spikes.

In [ ]:
for key, lab in (("btmf", "significant STRF, bandpass (a)"), ("btmf_b", "significant STRF, bandpass (b)"), ("btmf_full", "full STRF, bandpass (a)")):
    for rows, sl in VCSETS:
        E_, I_ = vcol(key, "vc_exc", rows), vcol(key, "vc_inh", rows)
        print(f"{lab}, {sl}: E vs I:", ut(E_, I_)); print("   E vs spikes:", ut(E_, col(key, "spike"))); print("   I vs spikes:", ut(I_, col(key, "spike")))

**Result: not reproduced.** EPSC and IPSC bTMFs are similar under every reading and event set (E-I p >= 0.17), and the 20 pA means (11.60 and 11.10 Hz) fall within tolerance, but EPSCs exceed spike bTMFs under none (p >= 0.261; spikes 9.34 Hz in the 37 neurons under D11 (a), 11.96 Hz under (b)). IPSCs exceed spikes only at 40 pA under D11 (a) (16.04 against 9.34 Hz, p = 0.0223).

### tab2-event-rate-excitation-higher
> EPSC event rates (3.72 +- 0.74 Hz) were much higher than IPSC event rates (1.06 +- 0.61 Hz).

In [ ]:
for rows, lab in VCSETS + ((vc40_pre, "40 pA, pre-stimulus s.d."),):
    print(lab, "E vs I:", ut(vcol("evrate", "vc_exc", rows), vcol("evrate", "vc_inh", rows)))

**Result: reproduced, with the pre-stimulus s.d. (D3 (a)) at 20 pA selectivity.** There EPSC rates (3.96 Hz) exceed IPSC rates (1.48 Hz, p = 0.00331), and both are within tolerance (3.72 +- 0.74, 1.06 +- 0.61). With the noise s.d. at 20 pA, excitation also exceeds inhibition (6.44 against 2.65 Hz, p = 0.0133), with both means outside tolerance. At 40 pA neither baseline gives a difference (1.31 against 1.28 Hz, p = 0.954; 1.14 against 0.78 Hz, p = 0.282). The match holds in one of the four event sets; the rates depend on the selectivity, which the Methods give only as about 20 to 40 pA.

### tab2-nonlinearity-ei
> EPSC nonlinearities had low thresholds (-1.14 +- 0.87 s.d.) and high transitions (2.21 +- 0.49),
> IPSC ones high thresholds (1.26 +- 0.70) and low transitions (1.03 +- 0.31); EPSC thresholds were
> below spikes and PSPs and IPSC thresholds above them; EPSC transitions exceeded spikes and small
> PSPs, IPSC transitions were similar to spikes and below PSPs.

In [ ]:
for key, ck in (("theta", "theta"), ("trans", "trans"), ("theta_full", "theta_full"), ("trans_full", "trans_full")):
    for rows, sl in VCSETS:
        E_, I_ = vcol(key, "vc_exc", rows), vcol(key, "vc_inh", rows)
        print(key, sl, "E vs I:", ut(E_, I_))
        for c in ("spike", "large", "small"):
            print(f"   E vs {c}:", ut(E_, col(ck, c))); print(f"   I vs {c}:", ut(I_, col(ck, c)))

**Result: not reproduced.** No event set and STRF reading puts the four means within tolerance with the E-I differences significant and enough of the other comparisons holding. The closest is the full STRF at 20 pA with the noise s.d.: EPSC threshold -1.51 and transition 2.34 and IPSC threshold 0.84 are within tolerance, the IPSC transition (1.41) is not, and E and I differ in threshold (p = 0.0115) and transition (p = 0.0364). But IPSC thresholds are below, not above, every current-clamp type, and IPSC transitions are above, not below, the PSPs', so 5 of the other 10 comparisons hold, where the partial grade needs 8. On the significant STRF at 20 pA, the IPSC threshold (0.07) and transition (1.43) fall outside tolerance and the E-I thresholds do not differ (p = 0.26).

### tab2-fsi-ei
> Feature selectivity indexes were slightly higher for IPSCs (0.13 +- 0.05) than for EPSCs (0.09
> +- 0.03); IPSC values did not differ from spikes and PSPs, while EPSC values were smaller than
> those of spikes and PSPs.

In [ ]:
for rows, sl in VCSETS:
    E_, I_ = vcol("fsi", "vc_exc", rows), vcol("fsi", "vc_inh", rows)
    print(sl, "E vs I:", ut(E_, I_))
    for c in ("spike", "large", "small"):
        print(f"   E vs {c}:", ut(E_, col("fsi", c))); print(f"   I vs {c}:", ut(I_, col("fsi", c)))

**Result: reproduced, with the pre-stimulus s.d. (D3 (a)) at 20 pA.** There the IPSC FSI (0.10) exceeds the EPSC FSI (0.07, p = 0.000938), both within tolerance; EPSCs are below spikes, large and small PSPs (p <= 0.0042), and IPSCs differ from none of the three (p = 0.163, 0.353 and 0.0512, the last near the threshold). With the noise s.d. at 20 pA both means lie 0.01 below their tolerances and IPSCs are below large and small PSPs (p <= 4.74e-05); at 40 pA IPSCs do not exceed EPSCs (0.11 against 0.13, p = 0.127). The match holds in one of the three event sets.

### tab2-latency-duration-ei-no-difference
> STRF latency (35.2 +- 10.9 against 43.5 +- 14.8 ms) and duration (41.5 +- 16.9 against 45.6 +-
> 29.0 ms) did not differ between excitation and inhibition.

In [ ]:
for rows, sl in VCSETS:
    for k in ("lat_a", "lat_b", "dur_a", "dur_b"):
        print(sl, k, "E vs I:", ut(vcol(k, "vc_exc", rows), vcol(k, "vc_inh", rows)))

**Result: not reproduced.** With peak latency (D10 (a)), inhibition is later than excitation in every event set: 51.1 against 33.0 ms at 20 pA (p = 0.00755), 60.5 against 39.0 ms at 40 pA (p = 0.0384) and 50.6 against 33.9 ms with the pre-stimulus s.d. (p = 0.0156). The earliest-pixel latency (b) does not differ (p >= 0.0969), nor do the durations (p >= 0.0849). Reading (a), the Results' "peak latency", is adopted, so the claim of absence fails. The 20 pA means under (a) are within tolerance.

## Supplementary Figs. 2 and 3: what the data set holds

The data set holds one neuron for each (the data description: "a representative example"),
so the three claims on them are blocked (data_missing). The archive's members under
`Supplementary/` are listed in the data check. The files are checked against the main
recordings, and the one pair and the one neuron are measured, as the criteria ask, without a
test.

In [ ]:
for k in ("supp2", "supp3"):
    for r in by[k]:
        twin = [x["file"] for x in by["cc_dmr"] if x["md5_raw"] == r["md5_raw"]]
        print(k, r["folder"].split("/")[-1], r["file"], "identical to main DMR recording:", twin)
# What the one neuron of each allows, as the criteria ask (not a test of the claims).
sup = {}
for r in by["supp2"] + by["supp3"]:
    ev = K.split_events(r, K.MAIN)
    sup[(r["kind"], r["folder"].split("/")[-1])] = K.analyse_strfs(r, envf, faxis, ev)
a, b = sup[("supp2", "repeat1")], sup[("supp2", "repeat2")]
print("Supplementary Fig. 2, the one pair present (repeat1 17921004, repeat2 17921007): correlation coefficient between the two STRFs")
for c in ("spike", "large", "small", "all"):
    print(f"  {c}: full STRF r = {np.corrcoef(a[c]['rf'].ravel(), b[c]['rf'].ravel())[0, 1]:.2f}; events {a[c]['n0']} and {b[c]['n0']}")
print("Supplementary Fig. 3, the one neuron at three levels: BF (kHz), peak latency (D10 (a), ms), total bandwidth (D8 (a), octaves)")
for lvl in ("38dBSPL", "54dBSPL", "69dBSPL"):
    s_ = sup[("supp3", lvl)]
    print("  " + lvl + ": " + "; ".join(f"{c} BF {s_[c].get('bf', np.nan) / 1000:.1f}, latency {s_[c].get('lat_a', np.nan):.0f}, bandwidth {s_[c].get('total_a', np.nan):.2f}"
                                    for c in ("spike", "large", "small")))

The one Supplementary Fig. 2 pair present gives full-STRF correlations of 0.18 (spikes), 0.12 (large PSPs), 0.02 (small PSPs) and 0.24 (all events); the paper's 13 pairs range from 0.30 to 0.76 for spikes. One pair cannot test either claim. The one Supplementary Fig. 3 neuron has a spike BF of 20.0 kHz at all three levels, with peak latencies of 45, 30 and 40 ms; its large- and small-PSP STRFs have no subfield at 69 dB SPL. Five neurons are needed for the ANOVA.

## Summary

| Claim | Outcome |
|---|---|
| cc-cohort-counts | reproduced (mice not countable) |
| vc-cohort-counts | reproduced (mice not countable) |
| fig1-psp-amplitude-kinetics | reproduced, assumed |
| fig1c-resting-potential-range | reproduced |
| fig1d-ndr-distribution | reproduced, assumed |
| fig1e-ndr-vs-rest-weak | reproduced, assumed |
| fig1fgh-trf-bandwidth-narrowing | partially reproduced |
| fig2-psp-histogram-modality | not reproduced, assumed |
| fig2-strf-significance-groups | not reproduced, assumed |
| fig2-psp-amplitude-ranges | not reproduced, assumed |
| supfig2-strf-reliability | blocked, data missing |
| supfig2-psp-more-reliable-than-spikes | blocked, data missing |
| supfig3-intensity-invariance | blocked, data missing |
| fig3d-double-peaked-counts | not reproduced, assumed |
| fig3e-harmonic-ratios | not reproduced, assumed |
| fig3f-bandwidth-single-vs-double | reproduced, assumed |
| fig4c-btmf-small-psp-higher | not reproduced, assumed |
| fig4d-bsmf-small-psp-higher | not reproduced, assumed |
| tab1-modulation-large-psp-like-spikes | reproduced, assumed |
| fig4b-bsmf-btmf-correlation | not reproduced, assumed |
| fig5a-cf-bf-correlation | not reproduced, assumed |
| fig5b-bf-same-across-events | reproduced, assumed |
| fig5c-bandwidth-relationships | not reproduced, assumed |
| tab1-single-band-small-psp-narrower | partially reproduced, assumed |
| tab1-total-bandwidth-no-difference | not reproduced, assumed |
| tab1-latency-duration | not reproduced, assumed |
| tab1-event-rate-no-difference | not reproduced, assumed |
| tab1-fsi-spikes-higher-than-large | reproduced, assumed |
| fig6b-tab1-threshold-spikes-higher | not reproduced, assumed |
| fig6c-tab1-transition-spikes-lower | not reproduced, assumed |
| fig6bc-no-correlation-psp-spike-nonlinearity | reproduced, assumed |
| fig6d-spike-threshold-transition-covary | partially reproduced, assumed |
| fig6efg-fsi-relationships | not reproduced, assumed |
| fig5df-strf-narrower-than-trf | not reproduced, assumed |
| fig5e-q-strf-higher | reproduced, assumed |
| fig7-current-amplitude-unimodal | partially reproduced, assumed |
| fig7c-ei-bf-matched | partially reproduced, assumed |
| fig7d-current-double-peaks | not reproduced, assumed |
| fig7e-tab2-inhibition-broader | reproduced, assumed |
| fig7f-tab2-ipsc-bsmf-lower | not reproduced, assumed |
| tab2-btmf-currents-higher-than-spikes | not reproduced, assumed |
| tab2-event-rate-excitation-higher | reproduced, assumed |
| tab2-nonlinearity-ei | not reproduced, assumed |
| tab2-fsi-ei | reproduced, assumed |
| tab2-latency-duration-ei-no-difference | not reproduced, assumed |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The tone and cohort findings use the paper's definitions and reproduce under the readings named. The STRF findings depend on choices the paper does not fix: the baseline s.d. that removes "very small events", how a histogram is judged bimodal, and how bands, bandwidths and double peaks are told from chance pixels. The pixel significance follows the Methods (p < 0.05 against circularly shifted event trains), where the shipped code uses an analytic threshold (p = 0.005). Both of the claims file's readings of a significant STRF select the same 37 neurons, against the paper's 30, and the Table 1 claims are graded on them; the code's analytic significance (29 neurons) and a cluster test on the shift null (15 neurons, the first round's group) are shown as variants, and several outcomes differ between the groups (bTMF, bandwidth, event rate, FSI, nonlinearity correlations). Bands within an STRF are subfields kept by the cluster test, an assumption, since without it nearly every STRF is double-peaked. STRFs use the downsampled envelope the paper's `wcr_strf.m` names (5 ms by 0.1 octave), which quantizes peak ratios and bandwidths. Voltage-clamp events use selectivity 20 pA and 40 pA, on the inverted current for excitation, with both baseline readings; three voltage-clamp claims reproduce only with the pre-stimulus s.d. at 20 pA. The TRF characteristic frequency, set at the 0 dB SPL threshold from single presentations, does not agree with the TRF's best frequency at higher levels, which may explain the failed CF-BF correlation.

What was found about the data set. Every recording the paper counts is present, and the data set's table pairs them as the paper describes. Two Supplementary recordings are byte-identical copies of main DMR recordings (17726007, 17921007). The Supplementary Figs. 2 and 3 folders hold one neuron each, not 2 and 5, as the listing of the archive's members shows. No animal identifier is given, so the mouse counts (41, 24) cannot be checked; the recordings fall on 39 and 22 dates. The data description gives processed = raw - smooth(raw, 1e5), but this holds for only 16 of 84 DMR recordings: 67 differ from it over most samples (largest differences 0.3 to 24 mV per recording) and one at 39 samples. Events here are detected on the processed traces, as the authors' derived data. Five excitatory recordings were held at -70 mV, not the -80 mV of the description (the table shows it). The data set ships the paper's analysis code, which settles the event detector, the STRF estimator, the nonlinearity fit and the FSI. It does not hold the code for event classification, subfields, bandwidth, BF, latency, RTF or the TRF.